In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
from matplotlib_venn import venn2
from tabulate import tabulate
import matplotlib.patheffects as path_effects
from plotly import graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
import plotly.express as px
import oracledb

In [4]:
dsn_tns = oracledb.makedsn("192.168.202.42", "1526", service_name="edwdb")  # Replace with actual details
dsn_tns_prod = oracledb.makedsn(host="192.168.202.75",port=1521,sid="icubeprd")
connection = oracledb.connect(user="AA3418KE", password="#Moving0n030@030", dsn=dsn_tns)
connection_prod = oracledb.connect(user="AA3418KE", password="#Moving0n030@030", dsn=dsn_tns_prod)

In [5]:
###solobiz ###

query = """
With GamCte as
(
select  
---acct_name as ACCOUNT_NAME,
foracid as ACCOUNT_NUMBER,cif_id as CUSTOMER_ID,
acct_crncy_code as account_currency,
CLR_BAL_AMT as ACCOUNT_BALANCE,
acct_cls_flg, ENTITY_CRE_FLG, DEL_FLG,
schm_code, ACCT_OWNERSHIP, SCHM_TYPE
from IMKE.gam
where schm_code NOT IN ('CAL29','CAL21','CAL19','CAL17','ODF09','ODL11') 
AND SCHM_TYPE IN ('ODA','CAA')
and acct_cls_flg!='Y' and ENTITY_CRE_FLG='Y' AND DEL_FLG!='Y'  
AND ACCT_OWNERSHIP='C'
),
VisAcc as 
(
select orgkey from vision.accounts a
where a.lead_source in ('COR','DSO')
),
CoreInt as
(
select orgkey from vision.coreinterface 
),
Demographics as 
(
select orgkey from VISION.DEMOGRAPHIC
where DEMOGRAPHICTYPE='Account' AND sourceofincome like 'BU%'
)
select 
---ACCOUNT_NAME, 
ACCOUNT_NUMBER, CUSTOMER_ID,
ACCOUNT_CURRENCY, ACCOUNT_BALANCE
from GamCte g
where CUSTOMER_ID IN (select orgkey from VisAcc)
and CUSTOMER_ID in (select orgkey from CoreInt)
and CUSTOMER_ID in (select orgkey from Demographics)

"""

###=== Execute query and load into DataFrame === ###
solobiz_df = pd.read_sql(query, con=connection,dtype={'CUSTOMER_ID': str})

solobiz_df.to_csv('solobiz_df.csv', index = False)

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_64484/2889314425.py:45: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [21]:
query_1 = """
select customerid, preferredphone
---- ,segmentcode, id_branch,branch_name, sector, subsector, OCCUPATION
from EDW.MDM_IMKE
--- where segmentcode IN ('BL03','BL05','5000','3100','5200','5100','5400','3000','3200')
"""

#### Execute query and load into DataFrame ###
cust_info_df = pd.read_sql(query_1, con=connection, dtype={'CUSTOMERID': str})

cust_info_df.to_csv('cust_info_df.csv', index = False)


/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_64484/868535383.py:9: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [17]:
query_staff = """
SELECT DISTINCT CIF_ID from TBAADM.GAM@FINACLE
where schm_code in ('CAL25', 'CAL26')
and acct_cls_flg = 'N'
"""

#### Execute query and load into DataFrame ###
staff_df = pd.read_sql(query_staff, con=connection, dtype={'CIF_ID': str})

staff_df.to_csv('staff_df.csv', index = False)

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_64484/3459019055.py:8: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [6]:
refresh_df = pd.read_csv('Latest_Limit_For_Upload_To_Redis_Mar202613_Raw_v2.csv',
                           dtype={'CIF_ID': str})
previous_df = pd.read_csv('Latest_Limit_For_Upload_To_Redis_Feb202613_Raw_v2.csv',
                           dtype={'CIF_ID': str})
solobiz_df = pd.read_csv('solobiz_df.csv',dtype={'CUSTOMER_ID': str})
crb_june = pd.read_csv('new_crb_jun2025.csv', dtype={'CIF_ID':str})
cust_info_df = pd.read_csv('cust_info_df.csv', dtype={'CUSTOMERID':str})

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/819725850.py:1: DtypeWarning:

Columns (49,68,72,73,74,79,118,152,153,154,162,164,172,173) have mixed types. Specify dtype option on import or set low_memory=False.

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/819725850.py:3: DtypeWarning:

Columns (153,154) have mixed types. Specify dtype option on import or set low_memory=False.



In [8]:
# Rename the column
solobiz_df.rename(columns={ 'CUSTOMER_ID': 'CIF_ID'}, inplace=True)
cust_info_df.rename(columns={ 'CUSTOMERID': 'CIF_ID'}, inplace=True)
solobiz_df.columns.to_list()

solobiz_df = solobiz_df[['CIF_ID']]

In [9]:
duplicates = solobiz_df[solobiz_df.duplicated(subset='CIF_ID', keep=False)]
dup_cust_df = cust_info_df[cust_info_df.duplicated(subset='CIF_ID', keep=False)]
print(len(duplicates))
print(len(dup_cust_df))
solobiz_df = solobiz_df.drop_duplicates(subset=['CIF_ID'], keep='first')
cust_info_df = cust_info_df.drop_duplicates(subset=['CIF_ID'], keep='first')
print(len(solobiz_df))
print(len(cust_info_df))

21347
0
129283
744182


In [10]:
refresh_df.shape

(524337, 179)

In [11]:
import os
# Get current working directory
current_path = os.getcwd()

print("Current notebook path:")
print(current_path)

Current notebook path:
/Users/ainea.ademba/Library/CloudStorage/OneDrive-I&MBankLimited/Documents/Digital Products Folder/All Digital Products/Digital Lending Products/STL/Limit Refreshes/May 2025


In [12]:
solobiz_df.shape

(129283, 1)

In [13]:
refresh_df.columns.to_list()

['CIF_ID',
 'CUST_AGE',
 'AGE_WITH_BANK',
 'PREDICTIONS',
 'RISK_GRADE',
 'CREDIT_SCORE',
 'PROXY_INCOME',
 'PROXY_INCOME_NEW',
 'PROXY_INCOME_MSE',
 'NUMBER_OF_TRANSACTING_MONTHS',
 'NUMBER_OF_20K_IN_4MONTHS',
 'COMPUTED_LIMIT',
 'LOYALTY_BAND',
 'NEXT_LOYALTY_BAND',
 'HYBRID_RISK_GRADE',
 'HYBRID_SCORE',
 'SEGMENT',
 'PROD_TENURE_UNSECURED_LONG_TERM',
 'PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR',
 'PROD_TENURE_UNSECURED_STL_DSR_MARGIN',
 'PROD_TENURE_OD_PRODUCT',
 'PROD_TENURE_BNPL_SHORT_TERM',
 'PROD_TENURE_BNPL_LONG_TERM',
 'D_I_RATIO_UNSECURED_LONG_TERM',
 'D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR',
 'D_I_RATIO_UNSECURED_STL_DSR_MARGIN',
 'D_I_RATIO_OD_PRODUCT',
 'D_I_RATIO_BNPL_SHORT_TERM',
 'D_I_RATIO_BNPL_LONG_TERM',
 'MIN_LIMIT_UNSECURED_LONG_TERM',
 'MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MIN_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MIN_LIMIT_OD_PRODUCT',
 'MIN_LIMIT_BNPL_SHORT_TERM',
 'MIN_LIMIT_BNPL_LONG_TERM',
 'MAX_LIMIT_UNSECURED_LONG_TERM',
 'MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MAX_LIMIT

In [14]:
refresh_df.head()

,CIF_ID,CUST_AGE,AGE_WITH_BANK,PREDICTIONS,RISK_GRADE,CREDIT_SCORE,PROXY_INCOME,PROXY_INCOME_NEW,PROXY_INCOME_MSE,NUMBER_OF_TRANSACTING_MONTHS,...,ELIGIBLE_EXSTAFF,LOYALTY_BAND_CURRENT,STL_PRICE_CAMPAIGN,CURRENT_LOYALTY_BAND,REAL_INELIGIBILITYREASON,CRB,DAYS_IN_BANK,NTB_NEW_STRATEGY,OD_TEST_STAFF,OD_TEST_STAFF_LIMIT
0,0433591,28,2,1,MEDIUM,588.0,17304.0,17304.0,17024.0,6,...,NaN,Platinum,0,NaN,NaN,NaN,NaN,NaN,{'short_term': {'message': 'Almost there! You ...,0
1,0370429,25,3,0,VERY HIGH,135.5,76742.0,76742.0,74992.0,6,...,NaN,Platinum,0,NaN,NaN,NaN,NaN,NaN,{'short_term': {'message': 'Almost there! You ...,0
2,0610917,29,1,1,MEDIUM,617.0,159302.5,159302.5,85018.0,6,...,NaN,Bronze,0,NaN,NaN,NaN,NaN,NaN,{'short_term': {'message': 'Almost there! You ...,0
3,0097494,53,11,1,LOW,827.0,49579.5,49579.5,49579.5,6,...,NaN,Platinum,0,NaN,NaN,NaN,NaN,NaN,{'short_term': {'message': 'Almost there! You ...,0
4,0170495,52,7,1,HIGH,413.5,29128.0,29128.0,29128.0,6,...,NaN,Platinum,0,NaN,NaN,NaN,NaN,NaN,{'short_term': {'message': 'Almost there! You ...,0


In [15]:
'CUST_3M_TENURE',
'LIMIT_3M_TENURE_ADJUSTED'

'LIMIT_3M_TENURE_ADJUSTED'

In [16]:
refresh_df_filter = refresh_df[['CIF_ID',
 'CUST_AGE',
 'AGE_WITH_BANK',
 'PREDICTIONS',
 'RISK_GRADE',
 'CREDIT_SCORE',
 'PROXY_INCOME',
 'PROXY_INCOME_NEW',
 'PROXY_INCOME_MSE',
 'NUMBER_OF_20K_IN_4MONTHS',
 'COMPUTED_LIMIT',
 'LOYALTY_BAND',
 'NEXT_LOYALTY_BAND',
 'HYBRID_RISK_GRADE',
 'HYBRID_SCORE',
 'SEGMENT',
 'PROD_TENURE_UNSECURED_LONG_TERM',
 'PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR',
 'PROD_TENURE_UNSECURED_STL_DSR_MARGIN',
 'PROD_TENURE_OD_PRODUCT',
 'PROD_TENURE_BNPL_SHORT_TERM',
 'PROD_TENURE_BNPL_LONG_TERM',
 'D_I_RATIO_UNSECURED_LONG_TERM',
 'D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR',
 'D_I_RATIO_UNSECURED_STL_DSR_MARGIN',
 'D_I_RATIO_OD_PRODUCT',
 'D_I_RATIO_BNPL_SHORT_TERM',
 'D_I_RATIO_BNPL_LONG_TERM',
 'MIN_LIMIT_UNSECURED_LONG_TERM',
 'MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MIN_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MIN_LIMIT_OD_PRODUCT',
 'MIN_LIMIT_BNPL_SHORT_TERM',
 'MIN_LIMIT_BNPL_LONG_TERM',
 'MAX_LIMIT_UNSECURED_LONG_TERM',
 'MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MAX_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MAX_LIMIT_OD_PRODUCT',
 'MAX_LIMIT_BNPL_SHORT_TERM',
 'MAX_LIMIT_BNPL_LONG_TERM',
 'BNPL_INTEREST_RATE',
 'UPL_INTEREST_RATE',
 'CUSTOMER_TYPE',
 'PRODUCT_INTERACTION',
 'CUSTOMER_RELATIONSHIP',
 'CREDIT_HISTORY',
 'MAX_DPD',
 'MAX_DPD_CRB',
 'SEGMENTATION_CLASS',
 'CRB_BORROWERS',
 'UPL_CONSERVATIVE_TENOR',
 'UPL_INTERMEDIATE_TENOR',
 'CREDIT_MARGIN',
 'UPL_CONSERVATIVE_DSR',
 'UPL_CONSERVATIVE_LIMIT',
 'UPL_INTERMEDIATE_DSR',
 'UPL_INTERMEDIATE_LIMIT',
 'UPL_CONSERVATIVE_INTEREST_RATE',
 'UPL_CONSERVATIVE_TENOR_MARGIN',
 'UPL_INTERMEDIATE_TENOR_MARGIN',
 'ADJUSTED_LIMIT',
 'INTEREST_FEE',
 'FACILITY_FEE',
 'CRB_NON_MOBILE_LOAN_HISTORY',
 'INTERNAL_LOAN_HISTORY',
 'FIRST_ACCT_OPN_DATE',
 'LAST_CLOSED_STL_OPN_DATE',
 'MONTHS_ACTIVE_STL',
 'MONTHS_STL_L6M',
 'OTGREGISTERED',
 'LAST_LOAN_OPN_DATE',
 'LAST_LOAN_CLS_DATE',
 'LAST_LOAN_DPD',
 'MAX_STL_DPD',
 'UPL_LIMIT',
 'TAKEN_UPL',
 'MAX_DPD_ALL_LOANS',
 'CLOSED_STL_OPENED>3M',
 'LAST_OPN_LOAN<3M',
 'LAST_CLS_LOAN<3M',
 'PERFORMING_INTERNALLY',
 'AGE_WITH_BANK_MONTHS',
 'NTB',
 'ARREARS0DAYS',
 'ARREARS30DAYS',
 'ARREARS60DAYS',
 'ARREARS90DAYS',
 'ARREARSGT90DAYS',
 'NONPERFORMING',
 'STL_ACTIVE_LOAN',
 'TAKEN_STL_LOAN',
 'STL_HISTORY',                            
 'LATEST_UTILIZED_LIMIT',
 'ID',
 'OPT_IN',
 'OPT_IN_DATE',
 'REDIS_LIMIT',
 'BLACKLISTED',
 'CREDIT_LIFE_INSURANCE',
'EXCISE_DUTY',
 'PHCF',
 'REFRESH_DATE',
 'NTB_LIMIT',
 'NTB_ADJUSTED_LIMIT',
 'CUST_3M_TENURE',
 'LIMIT_3M_TENURE',
 'LIMIT_3M_TENURE_ADJUSTED',
 'BNPL_MONTHLY_LIMIT_REDIS',
 'BNPL_TOTAL_LIMIT_REDIS',
 'UPL_MONTHLY_STP_LIMIT',
 'UPL_TOTAL_STP_LIMIT',
 'UPL_MONTHLY_NON_STP_LIMIT',
 'UPL_TOTAL_NON_STP_LIMIT',
 'BNPL_MONTHLY_LIMIT',
 'BNPL_TOTAL_LIMIT',
 'OTG_RET_LOGIN',
 'MSE_NON_SOLOBIZ', 
'STL_PRICE_CAMPAIGN',                              
 'BATCH_NO']]

In [27]:
refresh_df_filter['new_limits'] = ((refresh_df_filter['REDIS_LIMIT'] == 0) & (refresh_df_filter['ADJUSTED_LIMIT'] > 0)).astype(int)
refresh_df_filter['limit_increase'] = ((refresh_df_filter['new_limits'] == 0) & (refresh_df_filter['ADJUSTED_LIMIT'] > refresh_df_filter['REDIS_LIMIT'])).astype(int)
refresh_df_filter['limit_cancelled'] = (
    (refresh_df_filter['new_limits'] == 0) & 
    (refresh_df_filter['limit_increase'] == 0) & 
    (refresh_df_filter['ADJUSTED_LIMIT'] == 0) & 
    (refresh_df_filter['REDIS_LIMIT'] > 0)
).astype(int)
refresh_df_filter['limit_decrease'] = (
    (refresh_df_filter['new_limits'] == 0) & 
    (refresh_df_filter['limit_increase'] == 0) & 
    (refresh_df_filter['limit_cancelled'] == 0) & 
    (refresh_df_filter['ADJUSTED_LIMIT'] < refresh_df_filter['REDIS_LIMIT'])
).astype(int)



###BNPL limit perfomance Review 

refresh_df_filter['bnpl_new_limits'] = ((refresh_df_filter[ 'BNPL_TOTAL_LIMIT_REDIS'] == 0) & (refresh_df_filter[ 'BNPL_TOTAL_LIMIT'] > 0)).astype(int)
refresh_df_filter['bnpl_limit_increase'] = ((refresh_df_filter['bnpl_new_limits'] == 0) & (refresh_df_filter['BNPL_TOTAL_LIMIT'] > refresh_df_filter['BNPL_TOTAL_LIMIT_REDIS'])).astype(int)
refresh_df_filter['bnpl_limit_cancelled'] = (
    (refresh_df_filter['bnpl_new_limits'] == 0) & 
    (refresh_df_filter['bnpl_limit_increase'] == 0) & 
    (refresh_df_filter['BNPL_TOTAL_LIMIT'] == 0) & 
    (refresh_df_filter['BNPL_TOTAL_LIMIT_REDIS'] > 0)
).astype(int)
refresh_df_filter['bnpl_limit_decrease'] = (
    (refresh_df_filter['bnpl_new_limits'] == 0) & 
    (refresh_df_filter['bnpl_limit_increase'] == 0) & 
    (refresh_df_filter['bnpl_limit_cancelled'] == 0) & 
    (refresh_df_filter['BNPL_TOTAL_LIMIT'] < refresh_df_filter['BNPL_TOTAL_LIMIT_REDIS'])
).astype(int)

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/3512943021.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/3512943021.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/3512943021.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

In [29]:
merged_df = refresh_df_filter.merge(solobiz_df[['CIF_ID']], on='CIF_ID', how='left', indicator=True)

# Create the solo_flag column: 1 if CIF_ID is in solobiz_df, else 0
merged_df['solo_flag'] = (merged_df['_merge'] == 'both').astype(int)

#### Drop the _merge column (not needed anymore)
merged_df.drop(columns=['_merge'], inplace=True)

merged_df.head()

,CIF_ID,CUST_AGE,AGE_WITH_BANK,PREDICTIONS,RISK_GRADE,CREDIT_SCORE,PROXY_INCOME,PROXY_INCOME_NEW,PROXY_INCOME_MSE,NUMBER_OF_20K_IN_4MONTHS,...,BATCH_NO,new_limits,limit_increase,limit_cancelled,limit_decrease,bnpl_new_limits,bnpl_limit_increase,bnpl_limit_cancelled,bnpl_limit_decrease,solo_flag
0,0433591,28,2,1,MEDIUM,588.0,17304.0,17304.0,17024.0,4,...,202602,0,0,0,0,0,0,0,0,0
1,0370429,25,3,0,VERY HIGH,135.5,76742.0,76742.0,74992.0,4,...,202602,0,0,0,0,0,0,0,0,0
2,0610917,29,1,1,MEDIUM,617.0,159302.5,159302.5,85018.0,4,...,202602,0,0,0,0,0,0,0,0,0
3,0097494,53,11,1,LOW,827.0,49579.5,49579.5,49579.5,4,...,202602,0,0,0,0,0,0,0,0,0
4,0170495,52,7,1,HIGH,413.5,29128.0,29128.0,29128.0,4,...,202602,0,0,0,0,0,0,0,0,0


In [31]:
new_limits = merged_df['new_limits'].sum()
limit_increase = merged_df['limit_increase'].sum()
limit_cancelled = merged_df['limit_cancelled'].sum()
limit_decrease= merged_df['limit_decrease'].sum()



print("=" * 40)
print(f"📌 new_limits:  {new_limits:,.2f}")
print(f"📌 limit_increase:        {limit_increase:,.2f}")
print(f"📌 limit_cancelled:  {limit_cancelled:,.2f}")
print(f"📌 limit_decrease:        {limit_decrease:,.2f}")
print("=" * 40)


📌 new_limits:  13,646.00
📌 limit_increase:        1.00
📌 limit_cancelled:  697.00
📌 limit_decrease:        3.00


In [33]:
merged_df.columns.to_list()

['CIF_ID',
 'CUST_AGE',
 'AGE_WITH_BANK',
 'PREDICTIONS',
 'RISK_GRADE',
 'CREDIT_SCORE',
 'PROXY_INCOME',
 'PROXY_INCOME_NEW',
 'PROXY_INCOME_MSE',
 'NUMBER_OF_20K_IN_4MONTHS',
 'COMPUTED_LIMIT',
 'LOYALTY_BAND',
 'NEXT_LOYALTY_BAND',
 'HYBRID_RISK_GRADE',
 'HYBRID_SCORE',
 'SEGMENT',
 'PROD_TENURE_UNSECURED_LONG_TERM',
 'PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR',
 'PROD_TENURE_UNSECURED_STL_DSR_MARGIN',
 'PROD_TENURE_OD_PRODUCT',
 'PROD_TENURE_BNPL_SHORT_TERM',
 'PROD_TENURE_BNPL_LONG_TERM',
 'D_I_RATIO_UNSECURED_LONG_TERM',
 'D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR',
 'D_I_RATIO_UNSECURED_STL_DSR_MARGIN',
 'D_I_RATIO_OD_PRODUCT',
 'D_I_RATIO_BNPL_SHORT_TERM',
 'D_I_RATIO_BNPL_LONG_TERM',
 'MIN_LIMIT_UNSECURED_LONG_TERM',
 'MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MIN_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MIN_LIMIT_OD_PRODUCT',
 'MIN_LIMIT_BNPL_SHORT_TERM',
 'MIN_LIMIT_BNPL_LONG_TERM',
 'MAX_LIMIT_UNSECURED_LONG_TERM',
 'MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MAX_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MAX

In [35]:
# Merge the dataframes on 'CIF_ID', keeping all rows from refresh_df
merged_df = refresh_df_filter.merge(solobiz_df[['CIF_ID']], on='CIF_ID', how='left', indicator=True)

# Create the solo_flag column: 1 if CIF_ID is in solobiz_df, else 0
merged_df['solo_flag'] = (merged_df['_merge'] == 'both').astype(int)

# Drop the _merge column (not needed anymore)
merged_df.drop(columns=['_merge'], inplace=True)

# Filter rows where solo_flag is 1 and select desired columns
solo_filtered_df = merged_df[merged_df['solo_flag'] == 1][['CIF_ID', 'ADJUSTED_LIMIT']]

# Save to CSV
#solo_filtered_df.to_csv('solo_customers_limits.csv', index=False)

In [37]:
merged_df = merged_df.merge(previous_df[['CIF_ID', 'CUST_3M_TENURE', 'LIMIT_3M_TENURE_ADJUSTED']].rename(columns={
            'CUST_3M_TENURE': 'previous_3m',
            'LIMIT_3M_TENURE_ADJUSTED': 'limit_previous_3m'}),on='CIF_ID',how='left',indicator=True)
# Drop the _merge column (not needed anymore)
merged_df.drop(columns=['_merge'], inplace=True)

In [ ]:
###crb_june = pd.read_csv('new_crb_jun2025.csv', dtype={'CIF_ID':str})

In [ ]:
#merged_df[['CIF_ID']].to_excel('cif_ids_april refresh 19 May.xlsx', index = False)

In [39]:
# Merge the dataframes on 'CIF_ID', keeping all rows from refresh_df
previous_df_1 = previous_df.merge(solobiz_df[['CIF_ID']], on='CIF_ID', how='left', indicator=True)

# Create the solo_flag column: 1 if CIF_ID is in solobiz_df, else 0
previous_df_1['solo_flag'] = (previous_df_1['_merge'] == 'both').astype(int)

# Drop the _merge column (not needed anymore)
previous_df_1.drop(columns=['_merge'], inplace=True)



prv_stl_3m_customers = pd.to_numeric(previous_df[(previous_df['CUST_3M_TENURE'] == 1) & 
                                           (previous_df['LIMIT_3M_TENURE_ADJUSTED'] > 0)].CIF_ID.count(), errors='coerce')

prv_stl_3m_limit = pd.to_numeric(previous_df[previous_df['CUST_3M_TENURE'] == 1]['LIMIT_3M_TENURE_ADJUSTED'].sum(), errors='coerce')

prv_solobiz_stl_3m_customers = previous_df_1[
    (previous_df_1['CUST_3M_TENURE'] == 1) &
    (previous_df_1['solo_flag'] == 1) &
    (previous_df_1['LIMIT_3M_TENURE_ADJUSTED'] > 0)
]['CIF_ID'].nunique()  # Use count() if you want rows, nunique() for unique customers

# Sum of limits for same filtered group
prv_solobiz_stl_3m_limit = previous_df_1[
    (previous_df_1['CUST_3M_TENURE'] == 1) &
    (previous_df_1['solo_flag'] == 1)
]['LIMIT_3M_TENURE_ADJUSTED'].sum()



print(f"📌 PRV STL 3M Customers: {int(prv_stl_3m_customers):,}")  # Ensuring integer count
print(f"📌 PRV STL 3M Limit: {float(prv_stl_3m_limit):,.2f}")  # Ensuring float format with two decimal places

print(f"📌 SOLOBIZ PRV STL 3M Customers: {int(prv_solobiz_stl_3m_customers):,}")  
print(f"📌 SOLOBIZ PRV STL 3M Limit: {float(prv_solobiz_stl_3m_limit):,.2f}")


#merged_df['previous_3m'] = pd.to_numeric(previous_df['previous_3m'], errors='coerce').fillna(0).astype(float)
#merged_df['limit_previous_3m'] = pd.to_numeric(previous_df['limit_previous_3m'], errors='coerce').fillna(0).astype(float)

📌 PRV STL 3M Customers: 6,456
📌 PRV STL 3M Limit: 507,529,413.00
📌 SOLOBIZ PRV STL 3M Customers: 1,405
📌 SOLOBIZ PRV STL 3M Limit: 130,092,086.00


In [41]:
# Merge the dataframes on 'CIF_ID', keeping all rows from refresh_df
previous_df_1 = previous_df_1.merge(crb_june[['CIF_ID']], on='CIF_ID', how='left', indicator=True)

# Create the solo_flag column: 1 if CIF_ID is in solobiz_df, else 0
previous_df_1['crb_june_flag'] = (previous_df_1['_merge'] == 'both').astype(int)

# Drop the _merge column (not needed anymore)
previous_df_1.drop(columns=['_merge'], inplace=True)


prv_crb_stl_3m_customers = pd.to_numeric(previous_df_1[(previous_df_1['CUST_3M_TENURE'] == 1) & 
                                           (previous_df_1['LIMIT_3M_TENURE_ADJUSTED'] > 0) & (previous_df_1['crb_june_flag'] == 1)].CIF_ID.count(), errors='coerce')

prv_crb_stl_3m_limit = previous_df_1[(previous_df_1['CUST_3M_TENURE'] == 1) & (previous_df_1['crb_june_flag'] == 1)]['LIMIT_3M_TENURE_ADJUSTED'].sum()

prv_crb_solobiz_stl_3m_customers = previous_df_1[
    (previous_df_1['CUST_3M_TENURE'] == 1) &
    (previous_df_1['crb_june_flag'] == 1) & (previous_df_1['solo_flag'] == 1) &
    (previous_df_1['LIMIT_3M_TENURE_ADJUSTED'] > 0)
]['CIF_ID'].nunique()  # Use count() if you want rows, nunique() for unique customers

# Sum of limits for same filtered group
prv_crb_solobiz_stl_3m_limit = previous_df_1[
    (previous_df_1['CUST_3M_TENURE'] == 1) &
    (previous_df_1['crb_june_flag'] == 1) & (previous_df_1['solo_flag'] == 1)
]['LIMIT_3M_TENURE_ADJUSTED'].sum()

print(f"📌 PRV CRB STL 3M Customers: {int(prv_crb_stl_3m_customers):,}")  # Ensuring integer count
print(f"📌 PRV CRB STL 3M Limit: {float(prv_crb_stl_3m_limit):,.2f}")  # Ensuring float format with two decimal places

print(f"📌 SOLOBIZ CRB PRV STL 3M Customers: {int(prv_crb_solobiz_stl_3m_customers):,}")  
print(f"📌 SOLOBIZ CRB PRV STL 3M Limit: {float(prv_crb_solobiz_stl_3m_limit):,.2f}")


📌 PRV CRB STL 3M Customers: 1,360
📌 PRV CRB STL 3M Limit: 145,971,707.00
📌 SOLOBIZ CRB PRV STL 3M Customers: 339
📌 SOLOBIZ CRB PRV STL 3M Limit: 38,535,929.00


In [43]:
# Convert to numeric (if not already)
merged_df['previous_3m'] = pd.to_numeric(merged_df['previous_3m'], errors='coerce')
merged_df['limit_previous_3m'] = pd.to_numeric(merged_df['limit_previous_3m'], errors='coerce')

In [45]:
total_bnpl_new_limits = merged_df['bnpl_new_limits'].sum()
total_bnpl_limit_increase = merged_df['bnpl_limit_increase'].sum()
total_bnpl_limit_cancelled = merged_df['bnpl_limit_cancelled'].sum()
total_bnpl_limit_decrease= merged_df['bnpl_limit_decrease'].sum()



print("=" * 40)
print(f"📌 total_bnpl_new_limits:  {total_bnpl_new_limits:,.2f}")
print(f"📌 total_bnpl_limit_increase:        {total_bnpl_limit_increase:,.2f}")
print(f"📌 total_bnpl_limit_cancelled:  {total_bnpl_limit_cancelled:,.2f}")
print(f"📌 total_bnpl_limit_decrease:        {total_bnpl_limit_decrease:,.2f}")
print("=" * 40)


📌 total_bnpl_new_limits:  4.00
📌 total_bnpl_limit_increase:        0.00
📌 total_bnpl_limit_cancelled:  4.00
📌 total_bnpl_limit_decrease:        0.00


In [47]:
merged_df.shape

(524337, 130)

In [49]:
merged_df.tail()

,CIF_ID,CUST_AGE,AGE_WITH_BANK,PREDICTIONS,RISK_GRADE,CREDIT_SCORE,PROXY_INCOME,PROXY_INCOME_NEW,PROXY_INCOME_MSE,NUMBER_OF_20K_IN_4MONTHS,...,limit_increase,limit_cancelled,limit_decrease,bnpl_new_limits,bnpl_limit_increase,bnpl_limit_cancelled,bnpl_limit_decrease,solo_flag,previous_3m,limit_previous_3m
524332,0844775,27,0,0,VERY HIGH,303.0,12361.0,1030.083333,0.0,0,...,0,0,0,0,0,0,0,0,0.0,0.0
524333,0840394,22,0,0,VERY HIGH,140.0,16000.0,1333.333333,0.0,0,...,0,0,0,0,0,0,0,0,0.0,0.0
524334,0830146,19,0,0,VERY HIGH,144.0,650000.0,21666.666667,0.0,1,...,0,0,0,0,0,0,0,0,0.0,0.0
524335,0848064,34,0,0,HIGH,392.0,35000.0,2916.666667,0.0,1,...,0,0,0,0,0,0,0,0,0.0,0.0
524336,0844741,27,0,0,VERY HIGH,303.0,30000.0,2500.000000,0.0,1,...,0,0,0,0,0,0,0,0,0.0,0.0


In [51]:
### === 3 months loans limit check === ###

merged_df['3m_new_limits'] = ((merged_df['limit_previous_3m'] == 0) & (merged_df['LIMIT_3M_TENURE_ADJUSTED'] > 0)).astype(int)
merged_df['3m_limit_increase'] = ((merged_df['3m_new_limits'] == 0) &  (merged_df['LIMIT_3M_TENURE_ADJUSTED'] > merged_df['limit_previous_3m'])).astype(int)
merged_df['3m_limit_cancelled'] = ((merged_df['3m_new_limits'] == 0) & (merged_df['3m_limit_increase'] == 0) & 
                                       (merged_df['LIMIT_3M_TENURE_ADJUSTED'] == 0) & (merged_df['limit_previous_3m'] > 0)).astype(int)
merged_df['3m_limit_decrease'] = ((merged_df['3m_new_limits'] == 0) & (merged_df['3m_limit_increase'] == 0) & 
                 (merged_df['3m_limit_cancelled'] == 0) & (merged_df['LIMIT_3M_TENURE_ADJUSTED'] < merged_df['limit_previous_3m'])).astype(int)


In [53]:
total_3m_new_limits = merged_df['3m_new_limits'].sum()
total_3m_limit_increase = merged_df['3m_limit_increase'].sum()
total_3m_limit_cancelled = merged_df['3m_limit_cancelled'].sum()
total_3m_limit_decrease= merged_df['3m_limit_decrease'].sum()


print("=" * 40)
print(f"📌 total_3m_new_limits:  {total_3m_new_limits:,.2f}")
print(f"📌 total_3m_limit_increase:        {total_3m_limit_increase:,.2f}")
print(f"📌 total_3m_limit_cancelled:  {total_3m_limit_cancelled:,.2f}")
print(f"📌 total_3m_limit_decrease:        {total_3m_limit_decrease:,.2f}")
print("=" * 40)


📌 total_3m_new_limits:  874.00
📌 total_3m_limit_increase:        0.00
📌 total_3m_limit_cancelled:  91.00
📌 total_3m_limit_decrease:        0.00


In [55]:
['previous_3m']
'limit_previous_3m'

 #'STL_HISTORY',
 #  'OTG_RET_LOGIN',
# 'MSE_NON_SOLOBIZ',

prv_stl_3m_customers = pd.to_numeric(merged_df[(merged_df['previous_3m'] == 1) & 
                                           (merged_df['limit_previous_3m'] > 0)].CIF_ID.count(), errors='coerce')
prv_stl_3m_cust_solo = pd.to_numeric(merged_df[(merged_df['previous_3m'] == 1) & 
                                           (merged_df['limit_previous_3m'] > 0) & (merged_df['solo_flag'] == 1)].CIF_ID.count(), errors='coerce')
prv_stl_3m_cust_mse = pd.to_numeric(merged_df[(merged_df['previous_3m'] == 1) & 
                                           (merged_df['limit_previous_3m'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y')].CIF_ID.count(), errors='coerce')


prv_stl_3m_limit = pd.to_numeric(merged_df[merged_df['previous_3m'] == 1]['limit_previous_3m'].sum(), errors='coerce')
prv_stl_3m_lim_solo = (merged_df.loc[(merged_df['previous_3m'] == 1) & (merged_df['solo_flag'] == 1), 'limit_previous_3m'].astype(float).sum())
prv_stl_3m_lim_mse = (merged_df.loc[(merged_df['previous_3m'] == 1) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y'), 'limit_previous_3m'].astype(float).sum())

print(f"📌 PRV STL 3M Customers: {int(prv_stl_3m_customers):,}")  # Ensuring integer count
print(f"📌 PRV STL 3M Limit: {float(prv_stl_3m_limit):,.2f}")  # Ensuring float format with two decimal places
print(f"📌 PRV STL 3M Cust Solo: {int(prv_stl_3m_cust_solo):,}")  # Ensuring integer count
print(f"📌 PRV STL 3M Lim Solo: {float(prv_stl_3m_lim_solo):,.2f}")
print(f"📌 PRV STL 3M Cust MSE: {int(prv_stl_3m_cust_mse):,}")  # Ensuring integer count
print(f"📌 PRV STL 3M Lim MSE: {float(prv_stl_3m_lim_mse):,.2f}")


merged_df['previous_3m'] = pd.to_numeric(merged_df['previous_3m'], errors='coerce').fillna(0).astype(float)
merged_df['limit_previous_3m'] = pd.to_numeric(merged_df['limit_previous_3m'], errors='coerce').fillna(0).astype(float)

📌 PRV STL 3M Customers: 6,455
📌 PRV STL 3M Limit: 507,447,897.00
📌 PRV STL 3M Cust Solo: 1,405
📌 PRV STL 3M Lim Solo: 130,092,086.00
📌 PRV STL 3M Cust MSE: 646
📌 PRV STL 3M Lim MSE: 61,440,477.00


In [57]:
stl_3m_customers = pd.to_numeric(merged_df[(merged_df['CUST_3M_TENURE'] == 1) & 
                                           (merged_df['LIMIT_3M_TENURE_ADJUSTED'] > 0)].CIF_ID.count(), errors='coerce')
stl_3m_cust_mse = pd.to_numeric(merged_df[(merged_df['CUST_3M_TENURE'] == 1) & 
                                           (merged_df['LIMIT_3M_TENURE_ADJUSTED'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & 
                                (merged_df['OTG_RET_LOGIN'] == 'Y')].CIF_ID.count(), errors='coerce')

stl_3m_limit = pd.to_numeric(merged_df[merged_df['CUST_3M_TENURE'] == 1]['LIMIT_3M_TENURE_ADJUSTED'].sum(), errors='coerce')
stl_3m_lim_mse = pd.to_numeric(merged_df[(merged_df['CUST_3M_TENURE'] == 1) &
        (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y')]['LIMIT_3M_TENURE_ADJUSTED'],
    errors='coerce').sum()



print(f"📌 STL 3M Customers: {int(stl_3m_customers):,}")  # Ensuring integer count
print(f"📌 STL 3M Limit: {float(stl_3m_limit):,.2f}")  # Ensuring float format with two decimal places
print(f"📌 STL 3M Cust MSE: {int(stl_3m_cust_mse):,}")  # Ensuring integer count
print(f"📌 STL 3M Lim MSE: {float(stl_3m_lim_mse):,.2f}")  # Ensuring float format with two decimal places

merged_df['CUST_3M_TENURE'] = pd.to_numeric(merged_df['CUST_3M_TENURE'], errors='coerce').fillna(0).astype(float)
merged_df['LIMIT_3M_TENURE_ADJUSTED'] = pd.to_numeric(merged_df['LIMIT_3M_TENURE_ADJUSTED'], errors='coerce').fillna(0).astype(float)


📌 STL 3M Customers: 7,303
📌 STL 3M Limit: 559,791,614.00
📌 STL 3M Cust MSE: 710
📌 STL 3M Lim MSE: 65,953,307.00


In [59]:
(merged_df['MSE_NON_SOLOBIZ'] == 'Y' ).sum()
((merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['solo_flag'] == 1)).sum()

0

In [61]:
print(merged_df[merged_df['ADJUSTED_LIMIT'] > 0].CIF_ID.nunique())
# Count of STL Customers with ADJUSTED_LIMIT > 0
stl_customers = merged_df[merged_df['ADJUSTED_LIMIT'] > 0]['CIF_ID'].nunique()
stl_cust_mse = merged_df[(merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y')]['CIF_ID'].nunique()
stl_cust_mse_new = merged_df[(merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y') & (merged_df['STL_HISTORY'] == 'NO')]['CIF_ID'].nunique()
ntb_cust = merged_df[(merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['NTB'] == 'Y')]['CIF_ID'].nunique()
ntb_cust_above_10K = merged_df[(merged_df['ADJUSTED_LIMIT'] >= 10000) & (merged_df['NTB'] == 'Y')]['CIF_ID'].nunique()

# Total STL Limit (sum of ADJUSTED_LIMIT where it's greater than 0)
stl_limit = pd.to_numeric(merged_df.loc[merged_df['ADJUSTED_LIMIT'] > 0, 'ADJUSTED_LIMIT'], errors='coerce').sum()
stl_lim_mse = pd.to_numeric( merged_df.loc[
        (merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y'),'ADJUSTED_LIMIT'
    ],errors='coerce').sum()
stl_lim_mse_new = pd.to_numeric( merged_df.loc[
        (merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y') & (merged_df['STL_HISTORY'] == 'NO'),'ADJUSTED_LIMIT'
    ],errors='coerce').sum()
stl_mse_lim_total = stl_lim_mse + stl_3m_lim_mse

ntb_amount = pd.to_numeric( merged_df.loc[
        (merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['NTB'] == 'Y'),'ADJUSTED_LIMIT'
        ],errors='coerce').sum()


# Print formatted results
print(f"📌 STL Customers: {int(stl_customers):,}")  
print(f"📌 STL Limit: {float(stl_limit):,.2f}") 
print(f"📌 STL Cust MSE 1 month: {int(stl_cust_mse):,}")  
print(f"📌 STL Lim MSE 1 month: {float(stl_lim_mse):,.2f}") 
print(f"📌 STL Lim MSE combined(1&3): {float(stl_mse_lim_total):,.2f}") 
print(f"📌 STL Cust MSE new 1 month: {int(stl_cust_mse_new):,}")  
print(f"📌 STL Lim MSE new 1 month: {float(stl_lim_mse_new):,.2f}") 
print(f"📌 STL NTB 1 month: {int(ntb_cust):,}")  
print(f"📌 STL Lim NTB 1 month: {float(ntb_amount):,.2f}") 
print(f"📌 STL NTB above 10K: {int(ntb_cust_above_10K):,}")  

152845
📌 STL Customers: 152,845
📌 STL Limit: 1,878,813,904.00
📌 STL Cust MSE 1 month: 1,733
📌 STL Lim MSE 1 month: 32,384,672.00
📌 STL Lim MSE combined(1&3): 98,337,979.00
📌 STL Cust MSE new 1 month: 923
📌 STL Lim MSE new 1 month: 12,440,733.00
📌 STL NTB 1 month: 7,349
📌 STL Lim NTB 1 month: 20,265,322.00
📌 STL NTB above 10K: 0


#### MSE Extract File 

In [63]:
merged_df.columns.to_list()

['CIF_ID',
 'CUST_AGE',
 'AGE_WITH_BANK',
 'PREDICTIONS',
 'RISK_GRADE',
 'CREDIT_SCORE',
 'PROXY_INCOME',
 'PROXY_INCOME_NEW',
 'PROXY_INCOME_MSE',
 'NUMBER_OF_20K_IN_4MONTHS',
 'COMPUTED_LIMIT',
 'LOYALTY_BAND',
 'NEXT_LOYALTY_BAND',
 'HYBRID_RISK_GRADE',
 'HYBRID_SCORE',
 'SEGMENT',
 'PROD_TENURE_UNSECURED_LONG_TERM',
 'PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR',
 'PROD_TENURE_UNSECURED_STL_DSR_MARGIN',
 'PROD_TENURE_OD_PRODUCT',
 'PROD_TENURE_BNPL_SHORT_TERM',
 'PROD_TENURE_BNPL_LONG_TERM',
 'D_I_RATIO_UNSECURED_LONG_TERM',
 'D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR',
 'D_I_RATIO_UNSECURED_STL_DSR_MARGIN',
 'D_I_RATIO_OD_PRODUCT',
 'D_I_RATIO_BNPL_SHORT_TERM',
 'D_I_RATIO_BNPL_LONG_TERM',
 'MIN_LIMIT_UNSECURED_LONG_TERM',
 'MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MIN_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MIN_LIMIT_OD_PRODUCT',
 'MIN_LIMIT_BNPL_SHORT_TERM',
 'MIN_LIMIT_BNPL_LONG_TERM',
 'MAX_LIMIT_UNSECURED_LONG_TERM',
 'MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MAX_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MAX

In [65]:
filtered_mse = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) & 
    (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y')]

filtered_mse_phone = filtered_mse.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 
                  # 'SEGMENTCODE'
                 ]], on='CIF_ID',how='inner',indicator='info')

filtered_mse_phone[['CIF_ID', 'ADJUSTED_LIMIT','OTG_RET_LOGIN','OPT_IN','MSE_NON_SOLOBIZ','STL_HISTORY','PREFERREDPHONE','CUST_3M_TENURE','LIMIT_3M_TENURE_ADJUSTED']] \
    .to_excel('mse_cust_stl_lim_Dec_18_2025.xlsx', index=False)

In [67]:
merged_df.columns.to_list()

['CIF_ID',
 'CUST_AGE',
 'AGE_WITH_BANK',
 'PREDICTIONS',
 'RISK_GRADE',
 'CREDIT_SCORE',
 'PROXY_INCOME',
 'PROXY_INCOME_NEW',
 'PROXY_INCOME_MSE',
 'NUMBER_OF_20K_IN_4MONTHS',
 'COMPUTED_LIMIT',
 'LOYALTY_BAND',
 'NEXT_LOYALTY_BAND',
 'HYBRID_RISK_GRADE',
 'HYBRID_SCORE',
 'SEGMENT',
 'PROD_TENURE_UNSECURED_LONG_TERM',
 'PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR',
 'PROD_TENURE_UNSECURED_STL_DSR_MARGIN',
 'PROD_TENURE_OD_PRODUCT',
 'PROD_TENURE_BNPL_SHORT_TERM',
 'PROD_TENURE_BNPL_LONG_TERM',
 'D_I_RATIO_UNSECURED_LONG_TERM',
 'D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR',
 'D_I_RATIO_UNSECURED_STL_DSR_MARGIN',
 'D_I_RATIO_OD_PRODUCT',
 'D_I_RATIO_BNPL_SHORT_TERM',
 'D_I_RATIO_BNPL_LONG_TERM',
 'MIN_LIMIT_UNSECURED_LONG_TERM',
 'MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MIN_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MIN_LIMIT_OD_PRODUCT',
 'MIN_LIMIT_BNPL_SHORT_TERM',
 'MIN_LIMIT_BNPL_LONG_TERM',
 'MAX_LIMIT_UNSECURED_LONG_TERM',
 'MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MAX_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MAX

In [69]:

summary = pd.DataFrame({
    'LOYALTY_BAND': merged_df['LOYALTY_BAND'].unique()
})

# Get counts for positive limits
positive_counts = merged_df[merged_df['ADJUSTED_LIMIT'] > 0].groupby('LOYALTY_BAND').size()

# Get total counts
total_counts = merged_df.groupby('LOYALTY_BAND').size()

# Combine into a DataFrame
loyalty_summary = pd.DataFrame({
    'POSITIVE_LIMIT_CUSTOMERS': positive_counts,
    'TOTAL_CUSTOMERS': total_counts
}).fillna(0).reset_index()

# Calculate percentages
loyalty_summary['PERCENT_WITH_POSITIVE_LIMIT'] = (
    loyalty_summary['POSITIVE_LIMIT_CUSTOMERS'] / 
    loyalty_summary['TOTAL_CUSTOMERS'] * 100
).round(2)

# Sort if desired (e.g., by loyalty band or customer count)
loyalty_summary = loyalty_summary.sort_values('LOYALTY_BAND')

# Display with fancy grid
print(tabulate(loyalty_summary, 
               headers=['Loyalty Band', 'Customers (Limit > 0)', 'Total Customers', '% with Positive Limit'],
               tablefmt='fancy_grid',
               showindex=False,
               floatfmt=".0f"))

╒════════════════╤═════════════════════════╤═══════════════════╤═════════════════════════╕
│ Loyalty Band   │   Customers (Limit > 0) │   Total Customers │   % with Positive Limit │
╞════════════════╪═════════════════════════╪═══════════════════╪═════════════════════════╡
│ Bronze         │                  124021 │            494505 │                      25 │
├────────────────┼─────────────────────────┼───────────────────┼─────────────────────────┤
│ Gold           │                    6463 │              6681 │                      97 │
├────────────────┼─────────────────────────┼───────────────────┼─────────────────────────┤
│ Platinum       │                   10269 │             10532 │                      98 │
├────────────────┼─────────────────────────┼───────────────────┼─────────────────────────┤
│ Silver         │                   12092 │             12619 │                      96 │
╘════════════════╧═════════════════════════╧═══════════════════╧═════════════════════════╛

In [71]:

summary = pd.DataFrame({
    'LOYALTY_BAND': previous_df['LOYALTY_BAND'].unique()
})

# Get counts for positive limits
positive_counts = previous_df[previous_df['ADJUSTED_LIMIT'] > 0].groupby('LOYALTY_BAND').size()

# Get total counts
total_counts = previous_df.groupby('LOYALTY_BAND').size()

# Combine into a DataFrame
loyalty_summary = pd.DataFrame({
    'POSITIVE_LIMIT_CUSTOMERS': positive_counts,
    'TOTAL_CUSTOMERS': total_counts
}).fillna(0).reset_index()

# Calculate percentages
loyalty_summary['PERCENT_WITH_POSITIVE_LIMIT'] = (
    loyalty_summary['POSITIVE_LIMIT_CUSTOMERS'] / 
    loyalty_summary['TOTAL_CUSTOMERS'] * 100
).round(2)

# Sort if desired (e.g., by loyalty band or customer count)
loyalty_summary = loyalty_summary.sort_values('LOYALTY_BAND')

# Display with fancy grid
print(tabulate(loyalty_summary, 
               headers=['Loyalty Band', 'Customers (Limit > 0)', 'Total Customers', '% with Positive Limit'],
               tablefmt='fancy_grid',
               showindex=False,
               floatfmt=".0f"))

╒════════════════╤═════════════════════════╤═══════════════════╤═════════════════════════╕
│ Loyalty Band   │   Customers (Limit > 0) │   Total Customers │   % with Positive Limit │
╞════════════════╪═════════════════════════╪═══════════════════╪═════════════════════════╡
│ Bronze         │                  111182 │            478769 │                      23 │
├────────────────┼─────────────────────────┼───────────────────┼─────────────────────────┤
│ Gold           │                    6435 │              6705 │                      96 │
├────────────────┼─────────────────────────┼───────────────────┼─────────────────────────┤
│ Platinum       │                   10247 │             10562 │                      97 │
├────────────────┼─────────────────────────┼───────────────────┼─────────────────────────┤
│ Silver         │                   12049 │             12676 │                      95 │
╘════════════════╧═════════════════════════╧═══════════════════╧═════════════════════════╛

In [73]:
# Count the number of columns
num_columns = merged_df.shape[0]

print(f"Number of columns: {num_columns}")

Number of columns: 524337


In [75]:
#print(merged_df.dtypes)
merged_df['PROXY_INCOME'] = pd.to_numeric(merged_df['PROXY_INCOME'], errors='coerce').fillna(0).astype(float)
merged_df['PROXY_INCOME_NEW'] = pd.to_numeric(merged_df['PROXY_INCOME_NEW'], errors='coerce').fillna(0).astype(float)
merged_df['BNPL_TOTAL_LIMIT_REDIS'] = pd.to_numeric(merged_df['BNPL_TOTAL_LIMIT_REDIS'], errors='coerce').fillna(0).astype(int)
merged_df['BNPL_TOTAL_LIMIT'] = pd.to_numeric(merged_df['BNPL_TOTAL_LIMIT'], errors='coerce').fillna(0).astype(int)


total_sum_bnpl_redis = merged_df['BNPL_TOTAL_LIMIT_REDIS'].sum()
total_sum_bnpl = merged_df['BNPL_TOTAL_LIMIT'].sum()


print("=" * 40)
print(f"📌 Total BNPL Redis:  {total_sum_bnpl_redis:,.2f}")
print(f"📌 Total BNPL:        {total_sum_bnpl:,.2f}")
print("=" * 40)


📌 Total BNPL Redis:  106,100,000.00
📌 Total BNPL:        106,700,000.00


In [77]:
both_stl_bnpl_count = pd.to_numeric(merged_df[(merged_df['ADJUSTED_LIMIT'] > 0) & 
                                           (merged_df['BNPL_TOTAL_LIMIT'] > 0)].CIF_ID.count(), errors='coerce')

stl_greater_bnpl_amount = pd.to_numeric(
    merged_df.loc[(merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['BNPL_TOTAL_LIMIT'] > 0) & 
        (merged_df['ADJUSTED_LIMIT'] > merged_df['BNPL_TOTAL_LIMIT']), 'ADJUSTED_LIMIT'], errors='coerce').sum()
stl_greater_bnpl_count = merged_df.loc[(merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['BNPL_TOTAL_LIMIT'] > 0) & 
    (merged_df['ADJUSTED_LIMIT'] > merged_df['BNPL_TOTAL_LIMIT'])].shape[0]

# Filter rows where both ADJUSTED_LIMIT and BNPL_TOTAL_LIMIT > 0
filtered_df = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) & 
    (merged_df['BNPL_TOTAL_LIMIT'] > 0)
]

# Convert to numeric (if not already)
filtered_df['ADJUSTED_LIMIT'] = pd.to_numeric(filtered_df['ADJUSTED_LIMIT'], errors='coerce')
filtered_df['BNPL_TOTAL_LIMIT'] = pd.to_numeric(filtered_df['BNPL_TOTAL_LIMIT'], errors='coerce')

# Calculate the total sum of each column
total_adjusted_limit_both = filtered_df['ADJUSTED_LIMIT'].sum()
total_bnpl_limit_both = filtered_df['BNPL_TOTAL_LIMIT'].sum()
both_stl_bnpl_amount = total_bnpl_limit_both + total_adjusted_limit_both

print(f"Total ADJUSTED_LIMIT (where both > 0): {total_adjusted_limit_both}")
print(f"Total BNPL_TOTAL_LIMIT (where both > 0): {total_bnpl_limit_both}")

print("=" * 40)
print(f"📌 Both_stl_bnpl_limit_count:  {both_stl_bnpl_count:,}")
print(f"📌 stl_limit_both:  {total_adjusted_limit_both:,.2f}")
print(f"📌 bnpl_limit_both:  {total_bnpl_limit_both:,.2f}")
print(f"📌 stl>bnpl_count:  {stl_greater_bnpl_count:,}")
print(f"📌 stl>bnpl_amount:  {stl_greater_bnpl_amount:,}")
print("=" * 40)


Total ADJUSTED_LIMIT (where both > 0): 36908027.0
Total BNPL_TOTAL_LIMIT (where both > 0): 104700000
📌 Both_stl_bnpl_limit_count:  1,352
📌 stl_limit_both:  36,908,027.00
📌 bnpl_limit_both:  104,700,000.00
📌 stl>bnpl_count:  43
📌 stl>bnpl_amount:  2,973,383.0


/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/1622351333.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_72364/1622351333.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [79]:
# ===================== Data Processing =====================
# Filter rows where both limits are positive
filtered_df = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) & 
    (merged_df['BNPL_TOTAL_LIMIT'] > 0)
].copy()

# Convert to numeric safely
filtered_df['ADJUSTED_LIMIT'] = pd.to_numeric(filtered_df['ADJUSTED_LIMIT'], errors='coerce')
filtered_df['BNPL_TOTAL_LIMIT'] = pd.to_numeric(filtered_df['BNPL_TOTAL_LIMIT'], errors='coerce')

# Calculate metrics
both_stl_bnpl_count = filtered_df['CIF_ID'].nunique()
total_adjusted_limit_both = filtered_df['ADJUSTED_LIMIT'].sum()
total_bnpl_limit_both = filtered_df['BNPL_TOTAL_LIMIT'].sum()
both_stl_bnpl_amount = total_adjusted_limit_both + total_bnpl_limit_both

stl_greater_df = filtered_df[filtered_df['ADJUSTED_LIMIT'] > filtered_df['BNPL_TOTAL_LIMIT']]
stl_greater_count = stl_greater_df['CIF_ID'].nunique()
stl_greater_amount = stl_greater_df['ADJUSTED_LIMIT'].sum()

bnpl_greater_df = filtered_df[filtered_df['BNPL_TOTAL_LIMIT'] > filtered_df['ADJUSTED_LIMIT']]
bnpl_greater_count = bnpl_greater_df['CIF_ID'].nunique()
bnpl_greater_amount = bnpl_greater_df['ADJUSTED_LIMIT'].sum()

# ===================== Results Presentation =====================
results = [
    ["Metric", "Count", "Amount"],
    ["Both STL & BNPL limit", f"{both_stl_bnpl_count:,}", f"{both_stl_bnpl_amount:,.2f}"],
    ["- STL Limit", "-", f"{total_adjusted_limit_both:,.2f}"],
    ["- BNPL Limit", "-", f"{total_bnpl_limit_both:,.2f}"],
    ["STL greater BNPL", f"{stl_greater_count:,}", f"{stl_greater_amount:,.2f}"],
    ["BNPL greater STL", f"{bnpl_greater_count:,}", f"{bnpl_greater_amount:,.2f}"]
]

print(tabulate(results, headers="firstrow", tablefmt="fancy_grid", stralign="right", numalign="right"))
print("\n💡 STL = ADJUSTED_LIMIT | BNPL = BNPL_TOTAL_LIMIT")

╒═══════════════════════╤═════════╤════════════════╕
│                Metric │   Count │         Amount │
╞═══════════════════════╪═════════╪════════════════╡
│ Both STL & BNPL limit │   1,352 │ 141,608,027.00 │
├───────────────────────┼─────────┼────────────────┤
│           - STL Limit │       - │  36,908,027.00 │
├───────────────────────┼─────────┼────────────────┤
│          - BNPL Limit │       - │ 104,700,000.00 │
├───────────────────────┼─────────┼────────────────┤
│      STL greater BNPL │      43 │   2,973,383.00 │
├───────────────────────┼─────────┼────────────────┤
│      BNPL greater STL │   1,277 │  30,734,644.00 │
╘═══════════════════════╧═════════╧════════════════╛

💡 STL = ADJUSTED_LIMIT | BNPL = BNPL_TOTAL_LIMIT


In [81]:
# ===================== Data Processing =====================
# Filter rows where both limits are positive
bnpl_stl_history = merged_df[
    (merged_df['STL_HISTORY'] == 'YES') & 
    (merged_df['BNPL_TOTAL_LIMIT'] > 0)
].copy()

# Convert to numeric safely
filtered_df['BNPL_TOTAL_LIMIT'] = pd.to_numeric(filtered_df['BNPL_TOTAL_LIMIT'], errors='coerce')

# Calculate metrics
bnpl_stl_history_count = bnpl_stl_history['CIF_ID'].nunique()
bnpl_stl_history_limit_amount = bnpl_stl_history['BNPL_TOTAL_LIMIT'].sum()

# ===================== Results Presentation =====================
results = [
    ["Metric", "Count", "Amount"],
    ["BNPL with STL history", f"{bnpl_stl_history_count:,}", f"{bnpl_stl_history_limit_amount:,.2f}"]
]

print(tabulate(results, headers="firstrow", tablefmt="fancy_grid", stralign="right", numalign="right"))


╒═══════════════════════╤═════════╤═══════════════╕
│                Metric │   Count │        Amount │
╞═══════════════════════╪═════════╪═══════════════╡
│ BNPL with STL history │     668 │ 48,900,000.00 │
╘═══════════════════════╧═════════╧═══════════════╛


In [83]:
test = bnpl_stl_history['CIF_ID']
#test.to_excel('bnpl_with_stl 9 May.xlsx', index = False)

In [85]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: f'%.{len(str(x%1))-2}f' % x)
pd.set_option('display.max_colwidth', None)

##merged_df[merged_df['CIF_ID'] == '0184099']
previous_df[previous_df['CIF_ID'] == '0445912']

filtered_df = previous_df[previous_df['CIF_ID'].isin(['0414016'])].T
filtered_df

,5751
CIF_ID,0414016
CUST_AGE,34
AGE_WITH_BANK,2
PREDICTIONS,1
RISK_GRADE,MEDIUM
CREDIT_SCORE,697.0
PROXY_INCOME,572651.5
PROXY_INCOME_NEW,572651.5
PROXY_INCOME_MSE,572651.5
NUMBER_OF_TRANSACTING_MONTHS,6


In [87]:
previous_df[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN',	'OPT_IN_DATE']].head()
merged_df[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN',	'OPT_IN_DATE']].head()

previous_renamed = previous_df[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'OPT_IN_DATE']].copy()
previous_renamed = previous_renamed.rename(columns={
    'ADJUSTED_LIMIT': 'prv_ADJUSTED_LIMIT',
    'OPT_IN': 'prv_OPT_IN',
    'OPT_IN_DATE': 'prv_OPT_IN_DATE'
})


merged_selected = merged_df[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'OPT_IN_DATE']].copy()


combined_df = pd.merge(merged_selected, previous_renamed, on='CIF_ID', how='left')

combined_df.head()

,CIF_ID,ADJUSTED_LIMIT,OPT_IN,OPT_IN_DATE,prv_ADJUSTED_LIMIT,prv_OPT_IN,prv_OPT_IN_DATE
0,0433591,15536.0,1,2024-02-09 18:35:10,15536.0,1.0,2024-02-09 18:35:10
1,0370429,26731.0,0,2026-01-24 12:16:52,26731.0,0.0,2026-01-24 12:16:52
2,0610917,20000.0,1,2025-01-10 09:54:12,20000.0,1.0,2025-01-10 09:54:12
3,0097494,17889.0,1,2024-01-18 14:00:41,17889.0,1.0,2024-01-18 14:00:41
4,0170495,16291.0,1,2024-05-03 08:48:27,16291.0,1.0,2024-05-03 08:48:27


In [89]:
# Safely convert date column
combined_df['OPT_IN_DATE'] = pd.to_datetime(combined_df['OPT_IN_DATE'], errors='coerce')

# Base condition: prv_OPT_IN is 0 or NaN, and current OPT_IN is 1
base_condition = ((combined_df['prv_OPT_IN'].isna()) | (combined_df['prv_OPT_IN'] == 0)) & (combined_df['OPT_IN'] == 1)

# Count 1: total CIF_IDs meeting base condition
total_count = combined_df[base_condition]['CIF_ID'].nunique()

# Count 2: filter by April or May 2024
date_filtered = combined_df[
    base_condition &
    (combined_df['OPT_IN_DATE'].dt.month.isin([4, 5])) &
    (combined_df['OPT_IN_DATE'].dt.year == 2025)
]
april_may_count = date_filtered['CIF_ID'].nunique()

# Tabulate result
summary = {
    'Condition': [
        'prv_OPT_IN = 0 or NaN and current OPT_IN = 1',
        '...and OPT_IN_DATE in April or May 2025'
    ],
    'CIF_ID Count': [total_count, april_may_count]
}

print(tabulate(summary, headers='keys', tablefmt='fancy_grid', showindex=False))


╒══════════════════════════════════════════════╤════════════════╕
│ Condition                                    │   CIF_ID Count │
╞══════════════════════════════════════════════╪════════════════╡
│ prv_OPT_IN = 0 or NaN and current OPT_IN = 1 │           6848 │
├──────────────────────────────────────────────┼────────────────┤
│ ...and OPT_IN_DATE in April or May 2025      │              5 │
╘══════════════════════════════════════════════╧════════════════╛


In [91]:
combined_df['OPT_IN_DATE'] = pd.to_datetime(combined_df['OPT_IN_DATE'], errors='coerce')

filter_condition = (
    ((combined_df['prv_OPT_IN'].isna()) | (combined_df['prv_OPT_IN'] == 0)) &
    (combined_df['OPT_IN'] == 1) &
    (combined_df['OPT_IN_DATE'].dt.month.isin([4, 5])) &
    (combined_df['OPT_IN_DATE'].dt.year == 2025)
)

april_may_2025_df = combined_df[filter_condition]

#april_may_2025_df.to_excel('opted_in_april_may_2025.xlsx', index=False)

In [93]:
missing_in_refresh = previous_df[~previous_df['CIF_ID'].isin(refresh_df['CIF_ID'])]
missing_ids = missing_in_refresh['CIF_ID'].unique().tolist()

print(f"Number of CIF_IDs missing in refresh_df: {len(missing_ids)}")
#print(f"Missing CIF_IDs: {missing_ids}")

missing_ids_df = pd.DataFrame({'CIF_ID': missing_ids})

# Now perform the merge
missing_df = missing_ids_df.merge(previous_df, on='CIF_ID', how='left')
missing_df[['CIF_ID','CUST_AGE','AGE_WITH_BANK','RISK_GRADE', 'CREDIT_SCORE']].sample(10)

missing_df['CIF_ID'].to_excel('missing_CIF_IDs_29jan26.xlsx', index = False)

missing_df[missing_df['AGE_WITH_BANK'] > 11].head()

Number of CIF_IDs missing in refresh_df: 187


,CIF_ID,CUST_AGE,AGE_WITH_BANK,PREDICTIONS,RISK_GRADE,CREDIT_SCORE,PROXY_INCOME,PROXY_INCOME_NEW,PROXY_INCOME_MSE,NUMBER_OF_TRANSACTING_MONTHS,NUMBER_OF_20K_IN_4MONTHS,COMPUTED_LIMIT,LOYALTY_BAND,NEXT_LOYALTY_BAND,HYBRID_RISK_GRADE,HYBRID_SCORE,SEGMENT,PROD_TENURE_UNSECURED_LONG_TERM,PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR,PROD_TENURE_UNSECURED_STL_DSR_MARGIN,PROD_TENURE_OD_PRODUCT,PROD_TENURE_BNPL_SHORT_TERM,PROD_TENURE_BNPL_LONG_TERM,D_I_RATIO_UNSECURED_LONG_TERM,D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR,D_I_RATIO_UNSECURED_STL_DSR_MARGIN,D_I_RATIO_OD_PRODUCT,D_I_RATIO_BNPL_SHORT_TERM,D_I_RATIO_BNPL_LONG_TERM,MIN_LIMIT_UNSECURED_LONG_TERM,MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR,MIN_LIMIT_UNSECURED_STL_DSR_MARGIN,MIN_LIMIT_OD_PRODUCT,MIN_LIMIT_BNPL_SHORT_TERM,MIN_LIMIT_BNPL_LONG_TERM,MAX_LIMIT_UNSECURED_LONG_TERM,MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR,MAX_LIMIT_UNSECURED_STL_DSR_MARGIN,MAX_LIMIT_OD_PRODUCT,MAX_LIMIT_BNPL_SHORT_TERM,MAX_LIMIT_BNPL_LONG_TERM,BNPL_INTEREST_RATE,UPL_INTEREST_RATE,CUSTOMER_TYPE,PRODUCT_INTERACTION,CUSTOMER_RELATIONSHIP,CREDIT_HISTORY,MAX_DPD,MAX_DPD_CRB,SEGMENTATION_CLASS,CRB_BORROWERS,UPL_CONSERVATIVE_TENOR,UPL_INTERMEDIATE_TENOR,CREDIT_MARGIN,UPL_CONSERVATIVE_DSR,UPL_CONSERVATIVE_LIMIT,UPL_INTERMEDIATE_DSR,UPL_INTERMEDIATE_LIMIT,UPL_CONSERVATIVE_INTEREST_RATE,UPL_CONSERVATIVE_TENOR_MARGIN,UPL_INTERMEDIATE_TENOR_MARGIN,ADJUSTED_LIMIT,INTEREST_FEE,FACILITY_FEE,CRB_NON_MOBILE_LOAN_HISTORY,INTERNAL_LOAN_HISTORY,INTERNAL_LOAN_COUNT,FIRST_ACCT_OPN_DATE,LAST_CLOSED_STL_OPN_DATE,MONTHS_ACTIVE_STL,MONTHS_STL_L6M,OTGREGISTERED,LAST_LOAN_OPN_DATE,LAST_LOAN_CLS_DATE,LAST_ACCT_CLS_FLG,LAST_LOAN_DPD,MAX_STL_DPD_L6M,MAX_STL_DPD,UPL_LIMIT,TAKEN_UPL,MAX_DPD_ALL_LOANS,DPD_ARCHIVE,DPD_ARCHIVE_3M,DPD_ARCHIVE_6M,STL_3M_DIS_AMT,STL_3M_MAX_DIS_AMT,CLOSED_STL_OPENED>3M,LAST_OPN_LOAN<3M,LAST_CLS_LOAN<3M,PERFORMING_INTERNALLY,AGE_WITH_BANK_MONTHS,NTB,ARREARS0DAYS,ARREARS30DAYS,ARREARS60DAYS,ARREARS90DAYS,ARREARSGT90DAYS,NONPERFORMING,VALUE_CRB_LOAN_TAKEN,VOLUME_CRB_LOAN_TAKEN,MAXARREARSLAST12MONTHS,IM_LOAN_VALUE,IM_BASKET_SHARE,STL_ACTIVE_LOAN,TAKEN_STL_LOAN,STL_HISTORY,LATEST_UTILIZED_LIMIT,ID,OPT_IN,OPT_IN_DATE,REDIS_LIMIT,BNPL,UPL,OD,GLOBAL_LIMIT,LIMIT_3M_TENURE_REDIS,LIMIT_3M_TENURE_ADJUSTED_REDIS,CUST_3M_TENURE_REDIS,SEGMENTATION_CLASS_REDIS,LOYALTY_BAND_REDIS,NEXT_LOYALTY_BAND_REDIS,PREVIOUS_FACILITY_FEE,BLACKLISTED,WORKPLACE,GOV_WORKPLACE,SOLOBIZ,OTG_RET_LOGIN,MSE_NON_SOLOBIZ,BNPL_REDIS,UPL_REDIS,OD_REDIS,GLOBAL_LIMIT_REDIS,CREDIT_LIFE_INSURANCE,EXCISE_DUTY,PHCF,REFRESH_DATE,NTB_LIMIT,NTB_ADJUSTED_LIMIT,NEG_CAL_OD,CUST_3M_TENURE,LIMIT_3M_TENURE,APPLICABLE_LIMIT_3M_TOTAL,LIMIT_3M_TENURE_ADJUSTED,STL_3M_LOAN_TAKEN_AMT,STL_3M_LOAN_TAKEN,LIMIT_3M_ADJUSTED_ASSIGNED_REDIS,LIMIT_3M_TENURE_ADJUSTED_CURR,LIMIT_3M_STL_REPEAT,MONTHLY_LIMIT_3M,TENURES,BNPL_MONTHLY_LIMIT_REDIS,BNPL_TOTAL_LIMIT_REDIS,INELIGIBILITYREASON,EMPLOYEE_STATUS,GRADE,BASE_RATE,UPL_MONTHLY_STP_LIMIT,UPL_TOTAL_STP_LIMIT,UPL_MONTHLY_NON_STP_LIMIT,UPL_TOTAL_NON_STP_LIMIT,BNPL_MONTHLY_LIMIT,BNPL_TOTAL_LIMIT,BATCH_NO,LOYALTYBANDS,ELIGIBLE_EXSTAFF,CURRENT_LOYALTY_BAND,LOYALTY_BAND_CURRENT,OD_LIMIT,STL_PRICE_CAMPAIGN,REAL_INELIGIBILITYREASON
22,0060675,14,14,0,VERY HIGH,169.0,0.0,0.0,0.0,0,0,0.0,Bronze,Silver,VERY HIGH,91.0,Standard Credit,36,36,1,1,6,3,0.15,0.2,0.2,0.2,0.2,0.12,50000,50000,1000,5000,5000,0,2000000,2000000,100000,100000,100000,0,0.02,NaN,Premium,Medium,High,Weak,0,0,3100,Weak,12,24,8.0,0.0,800000,0.0,2000000,0.238,0.5,0.75,0.0,0.125,0.125,NO,NO,0,2011-10-25,NaN,0.0,0.0,N,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,0,0,0,0,0,0,N,N,N,N,174,N,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NO,NO,NO,0.0,NaN,0,NaN,0.0,"{'short_term': {'message': ""Sorry, you're currently not eligible for this loan product. Please try applying for other loan products. Enquiries, call 0719 088 000. Ref 014"", 'tenure': 0, 'monthly_limit': 0, 'applicable_limit': 0, 'interest_rate': 0, 'credit_life_insurance': 0, 'facility_fee': 0, 'processing_fee': 0, 'excise_duty': 0, 'phcf': 0}, 'long_term': {'message': ""Sorry, you'

In [95]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: f'%.{len(str(x%1))-2}f' % x)
pd.set_option('display.max_colwidth', None)

##merged_df[merged_df['CIF_ID'] == '0184099']
merged_df[merged_df['CIF_ID'] == '0382532']

filtered_df = refresh_df[refresh_df['CIF_ID'].isin(["0414016"])].T
filtered_df
#filtered_df[['CIF_ID', 'BNPL_TOTAL_LIMIT','AGE_WITH_BANK']]

,3150
CIF_ID,0414016
CUST_AGE,34
AGE_WITH_BANK,2
PREDICTIONS,1
RISK_GRADE,MEDIUM
CREDIT_SCORE,697.0
PROXY_INCOME,566455.5
PROXY_INCOME_NEW,566455.5
PROXY_INCOME_MSE,566455.5
NUMBER_OF_TRANSACTING_MONTHS,6


In [97]:
merged_df.to_csv('stl_bnpl_19_Mar_26.csv', index=False)

In [ ]:
#merged_df[['CIF_ID', 'SEGMENT','HYBRID_RISK_GRADE']].to_csv('stl_bnpl_seg_risk_sep_25.csv', index=False)

In [109]:
refresh_df[['CIF_ID','ADJUSTED_LIMIT', 'REAL_INELIGIBILITYREASON']].to_excel('stl_cust_limit_19_mar_2026_contact_centre.xlsx', index=False)

refresh_df[(refresh_df['ADJUSTED_LIMIT'] > 0) & (refresh_df['CUST_3M_TENURE'] == 1 )&
            (refresh_df['LIMIT_3M_TENURE_ADJUSTED'] > 0) ][['CIF_ID', 'LIMIT_3M_TENURE_ADJUSTED', 'CUST_3M_TENURE']].to_excel(
    '3months_stl_cust_limit_19_mar_2026_contact_centre.xlsx', index= False)

In [99]:
refresh_df.columns.to_list()  

['CIF_ID',
 'CUST_AGE',
 'AGE_WITH_BANK',
 'PREDICTIONS',
 'RISK_GRADE',
 'CREDIT_SCORE',
 'PROXY_INCOME',
 'PROXY_INCOME_NEW',
 'PROXY_INCOME_MSE',
 'NUMBER_OF_TRANSACTING_MONTHS',
 'NUMBER_OF_20K_IN_4MONTHS',
 'COMPUTED_LIMIT',
 'LOYALTY_BAND',
 'NEXT_LOYALTY_BAND',
 'HYBRID_RISK_GRADE',
 'HYBRID_SCORE',
 'SEGMENT',
 'PROD_TENURE_UNSECURED_LONG_TERM',
 'PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR',
 'PROD_TENURE_UNSECURED_STL_DSR_MARGIN',
 'PROD_TENURE_OD_PRODUCT',
 'PROD_TENURE_BNPL_SHORT_TERM',
 'PROD_TENURE_BNPL_LONG_TERM',
 'D_I_RATIO_UNSECURED_LONG_TERM',
 'D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR',
 'D_I_RATIO_UNSECURED_STL_DSR_MARGIN',
 'D_I_RATIO_OD_PRODUCT',
 'D_I_RATIO_BNPL_SHORT_TERM',
 'D_I_RATIO_BNPL_LONG_TERM',
 'MIN_LIMIT_UNSECURED_LONG_TERM',
 'MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MIN_LIMIT_UNSECURED_STL_DSR_MARGIN',
 'MIN_LIMIT_OD_PRODUCT',
 'MIN_LIMIT_BNPL_SHORT_TERM',
 'MIN_LIMIT_BNPL_LONG_TERM',
 'MAX_LIMIT_UNSECURED_LONG_TERM',
 'MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR',
 'MAX_LIMIT

##### Staff 

In [101]:
staff_df = pd.read_csv('staff_df.csv', dtype={'CIF_ID': str})

In [102]:
staff_df_merged = staff_df.merge(merged_df[['CIF_ID', 'BNPL_TOTAL_LIMIT', 'CUST_3M_TENURE','LIMIT_3M_TENURE_ADJUSTED']], on='CIF_ID', how='left')
staff_df_merged = staff_df_merged[(staff_df_merged['CUST_3M_TENURE'] > 0) & (staff_df_merged['LIMIT_3M_TENURE_ADJUSTED'] > 0)]
print(len(staff_df_merged))
#staff_df_merged[staff_df_merged['CIF_ID'] == '0114029']
staff_df_merged.head()
#staff_df_merged.to_excel('merged_staff_with_limits.xlsx', index = False)


399


,CIF_ID,BNPL_TOTAL_LIMIT,CUST_3M_TENURE,LIMIT_3M_TENURE_ADJUSTED
17,0003613,0.0,1.0,150000.0
51,0016214,150000.0,1.0,107689.0
55,0016434,150000.0,1.0,149688.0
67,0018460,100000.0,1.0,150000.0
79,0020957,100000.0,1.0,108632.0


##### MSE

In [123]:
staff_df_merged_prv = staff_df.merge(previous_df[['CIF_ID', 'BNPL_TOTAL_LIMIT', 'CUST_3M_TENURE','LIMIT_3M_TENURE_ADJUSTED']], on='CIF_ID', how='left')
staff_df_merged_prv = staff_df_merged_prv[(staff_df_merged_prv['CUST_3M_TENURE'] > 0) & (staff_df_merged['LIMIT_3M_TENURE_ADJUSTED'] > 0)]
#print(len(staff_df_merged_prv))
staff_df_merged[staff_df_merged['CIF_ID'] == '0082296']
#staff_df_merged_prv.head()
#staff_df_merged.to_excel('merged_staff_with_limits.xlsx', index = False)

,CIF_ID,BNPL_TOTAL_LIMIT,CUST_3M_TENURE,LIMIT_3M_TENURE_ADJUSTED


In [127]:
filtered_mse = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) & 
    (merged_df['MSE_NON_SOLOBIZ'] == 'Y') & (merged_df['OTG_RET_LOGIN'] == 'Y')]

filtered_mse_phone = filtered_mse.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 
                 # 'SEGMENTCODE'
                 ]], on='CIF_ID',how='inner',indicator='info')

filtered_mse_phone[['CIF_ID', 'ADJUSTED_LIMIT','OTG_RET_LOGIN','OPT_IN','MSE_NON_SOLOBIZ','STL_HISTORY','PREFERREDPHONE']] \
    .to_excel('mse_cust_stl_lim_19_Mar_26.xlsx', index=False)

In [129]:
mse_3_stl = pd.read_csv('mse_3m_limit_target_final_final 1.csv', dtype={'CIF_ID': str})

filtered_mse_3_stl = mse_3_stl[
    (mse_3_stl['ADJUSTED_LIMIT'] > 0) & 
    (mse_3_stl['MSE_NON_SOLOBIZ'] == 'Y') & 
    (mse_3_stl['OTG_RET_LOGIN'] == 'Y')
]

##### STL Base Campaign

In [131]:
#stl_cust_limsep26 = pd.read_excel('STL CIFS.xlsx', dtype={'CIF_ID': str})

query_open = """
select g.cif_id from TBAADM.GAM@FINACLE g
where SCHM_CODE in ('LAL34', 'LAL47')
and acct_cls_flg = 'N'
"""

#### Execute query and load into DataFrame ###
stl_cust_open = pd.read_sql(query_open, con=connection, dtype={'CIF_ID': str})

stl_cust_open.to_csv('stl_cust_open.csv', index = False)

stl_cust_open = pd.read_csv('stl_cust_open.csv', dtype={'CIF_ID': str})


/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_64484/2367225752.py:10: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [135]:
filtered_stl_camp = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['MSE_NON_SOLOBIZ'] == 'N') & (merged_df['NTB'] == 'N')]


filtered_stl_camp_phone = filtered_stl_camp.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 
                  # 'SEGMENTCODE'
                 ]], on='CIF_ID',how='inner',indicator='info')


filtered_3_stl_camp_phone = filtered_stl_camp_phone[(filtered_stl_camp_phone['ADJUSTED_LIMIT'] > 0) & (filtered_stl_camp_phone['CUST_3M_TENURE'] == 1 )&
                                                                            (filtered_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'] > 0) ]

#filtered_3_stl_camp_phone = filtered_3_stl_camp_phone[
    #filtered_3_stl_camp_phone['CIF_ID'].isin(stl_cust_limsep26['CIF_ID'])]

filtered_3_stl_camp_phone = filtered_3_stl_camp_phone[
    ~filtered_3_stl_camp_phone['CIF_ID'].isin(stl_cust_open['CIF_ID'])]

filtered_3_stl_camp_phone = filtered_3_stl_camp_phone[
    ~filtered_3_stl_camp_phone['CIF_ID'].isin(staff_df['CIF_ID'])]

filtered_3_stl_camp_phone = filtered_3_stl_camp_phone[
    filtered_3_stl_camp_phone['PREFERREDPHONE'].astype(str).str.startswith(('254', '+254'))
]

filtered_3_stl_camp_phone[['CIF_ID', 'CUST_3M_TENURE','LIMIT_3M_TENURE_ADJUSTED','OPT_IN','PREFERREDPHONE']] \
    .to_excel('19_Mar_2026_3mths.xlsx', index=False)

filtered_stl_camp_phone = filtered_stl_camp_phone[
    ~filtered_stl_camp_phone['CIF_ID'].isin(filtered_3_stl_camp_phone['CIF_ID'])]

#filtered_stl_camp_phone = filtered_stl_camp_phone[
    #filtered_stl_camp_phone['CIF_ID'].isin(stl_cust_limsep26['CIF_ID'])]

filtered_stl_camp_phone = filtered_stl_camp_phone[
    ~filtered_stl_camp_phone['CIF_ID'].isin(stl_cust_open['CIF_ID'])]

filtered_stl_camp_phone = filtered_stl_camp_phone[
    ~filtered_stl_camp_phone['CIF_ID'].isin(staff_df['CIF_ID'])]


filtered_stl_camp_phone = filtered_stl_camp_phone[
    filtered_stl_camp_phone['PREFERREDPHONE'].astype(str).str.startswith(('254', '+254'))
]

filtered_stl_camp_phone[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE','STL_HISTORY']] \
    .to_excel('19_Mar_2026_1mth.xlsx', index=False)


In [137]:

staff_customers_staff = merged_df[(merged_df['ADJUSTED_LIMIT'] > 0) & (merged_df['CIF_ID'].isin(staff_df['CIF_ID']))]['CIF_ID'].nunique()


staff_lim_stl = pd.to_numeric( merged_df.loc[
        (merged_df['ADJUSTED_LIMIT'] > 0)& (merged_df['CIF_ID'].isin(staff_df['CIF_ID'])),'ADJUSTED_LIMIT'
    ],errors='coerce').sum()



print(f"📌 STAFF STL: {int(staff_customers_staff):,}")  
print(f"📌 STAFF Limit: {float(staff_lim_stl):,.2f}") 


📌 STAFF STL: 1,928
📌 STAFF Limit: 48,736,232.00


In [139]:

staff_filtered_df = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) &
    (merged_df['CIF_ID'].isin(staff_df['CIF_ID']))
].copy()

# Ensure ADJUSTED_LIMIT is numeric
staff_filtered_df['ADJUSTED_LIMIT'] = pd.to_numeric(
    staff_filtered_df['ADJUSTED_LIMIT'],
    errors='coerce'
)

In [141]:
summary_table = (
    staff_filtered_df
    .groupby(['RISK_GRADE', 'SEGMENT'])
    .agg(
        CUSTOMER_COUNT=('CIF_ID', 'nunique'),
        TOTAL_AMOUNT=('ADJUSTED_LIMIT', 'sum')
    )
    .reset_index()
)

summary_table

,RISK_GRADE,SEGMENT,CUSTOMER_COUNT,TOTAL_AMOUNT
0,HIGH,Basic Credit,21,59527.0
1,HIGH,Premium Credit,98,3432885.0
2,HIGH,Standard Credit,70,1321202.0
3,LOW,Basic Credit,56,812926.0
4,LOW,Premium Credit,795,25391348.0
5,LOW,Standard Credit,111,1921184.0
6,MEDIUM,Basic Credit,27,465882.0
7,MEDIUM,Premium Credit,109,3043346.0
8,MEDIUM,Standard Credit,277,4816771.0
9,VERY HIGH,Basic Credit,27,185009.0


###### Dormant STL customers to target 

In [143]:
query_max = """
SELECT 
    g.cif_id,
    MAX(lht.applicable_date) AS disb_date
FROM TBAADM.GAM@FINACLE g
LEFT JOIN TBAADM.LHT@FINACLE lht 
    ON g.acid = lht.acid
WHERE g.schm_code IN ('LAL34', 'LAL47')
---and g.cif_id = '0376200'
AND lht.applicable_date < DATE '2026-01-01'
GROUP BY g.cif_id
"""

#### Execute query and load into DataFrame ###
stl_cust_open_max = pd.read_sql(query_max, con=connection, dtype={'CIF_ID': str})

stl_cust_open_max.to_csv('stl_cust_open_max.csv', index = False)



/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_64484/3746581676.py:15: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [144]:
one_month_target = filtered_stl_camp_phone[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE','STL_HISTORY']] 

one_month_target_dec_last_active = one_month_target[
    one_month_target['CIF_ID'].isin(stl_cust_open_max['CIF_ID'])]

one_month_target_dec_last_active = one_month_target_dec_last_active.merge(
    stl_cust_open_max[['CIF_ID', 'DISB_DATE']], on='CIF_ID'
    ,how='left',indicator='last_open')

one_month_target_dec_last_active.to_excel('1_month_stl_target_last_disb_dec_2025_19_mar.xlsx', index = False)

In [145]:
three_month_target = filtered_3_stl_camp_phone[['CIF_ID', 'CUST_3M_TENURE','LIMIT_3M_TENURE_ADJUSTED','OPT_IN','PREFERREDPHONE']]

three_month_target_dec_last_active = three_month_target[
    three_month_target['CIF_ID'].isin(stl_cust_open_max['CIF_ID'])]

three_month_target_dec_last_active = three_month_target_dec_last_active.merge(
    stl_cust_open_max[['CIF_ID', 'DISB_DATE']], on='CIF_ID'
    ,how='left',indicator='last_open')

three_month_target_dec_last_active.to_excel('3_month_stl_target_last_disb_dec_2025_19_mar.xlsx', index = False)

In [149]:
solobiz_1_month_df = filtered_stl_camp_phone[filtered_stl_camp_phone['solo_flag'] == 1]

solobiz_1_month_df.shape

solobiz_1_month_df[['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE','STL_HISTORY', 'solo_flag']] \
   .to_excel('19_Mar_2026_1mth_solobiz.xlsx', index=False)

In [171]:
# Basic summary info
summary_data = {
    "Metric": [
        "Row Count",
        "Unique CIF_IDs",
        #"Unique Phone Numbers",
        "Sum ADJUSTED_LIMIT",
        "Mean ADJUSTED_LIMIT",
        "Min ADJUSTED_LIMIT",
        "Max ADJUSTED_LIMIT",
        "OPT_IN (Yes count)"
    ],
    "filtered_stl_camp_phone": [
        len(filtered_stl_camp_phone),
        filtered_stl_camp_phone['CIF_ID'].nunique(),
        #filtered_stl_camp_phone['PREFERREDPHONE'].nunique(),
        f"{filtered_3_stl_camp_phone['ADJUSTED_LIMIT'].sum() / 1_000_000:,.2f}Mn",
        filtered_stl_camp_phone['ADJUSTED_LIMIT'].mean(),
        filtered_stl_camp_phone['ADJUSTED_LIMIT'].min(),
        filtered_stl_camp_phone['ADJUSTED_LIMIT'].max(),
        (filtered_stl_camp_phone['OPT_IN'] == 1).sum()
    ],
    "filtered_3_stl_camp_phone": [
        len(filtered_3_stl_camp_phone),
        filtered_3_stl_camp_phone['CIF_ID'].nunique(),
        #filtered_3_stl_camp_phone['PREFERREDPHONE'].nunique(),
        f"{filtered_3_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'].sum() / 1_000_000:,.2f}Mn",
        filtered_3_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'].mean(),
        filtered_3_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'].min(),
        filtered_3_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'].max(),
        (filtered_3_stl_camp_phone['OPT_IN'] == 1).sum()
    ]
}

# Convert to DataFrame for tabulation
summary_df = pd.DataFrame(summary_data)


print(tabulate(summary_df, headers='keys', tablefmt='fancy_grid', showindex=False))

╒═════════════════════╤═══════════════════════════╤═════════════════════════════╕
│ Metric              │ filtered_stl_camp_phone   │ filtered_3_stl_camp_phone   │
╞═════════════════════╪═══════════════════════════╪═════════════════════════════╡
│ Row Count           │ 97440                     │ 1337                        │
├─────────────────────┼───────────────────────────┼─────────────────────────────┤
│ Unique CIF_IDs      │ 97440                     │ 1337                        │
├─────────────────────┼───────────────────────────┼─────────────────────────────┤
│ Sum ADJUSTED_LIMIT  │ 55.17Mn                   │ 96.53Mn                     │
├─────────────────────┼───────────────────────────┼─────────────────────────────┤
│ Mean ADJUSTED_LIMIT │ 10458.533405172413        │ 72198.99775617053           │
├─────────────────────┼───────────────────────────┼─────────────────────────────┤
│ Min ADJUSTED_LIMIT  │ 1001.0                    │ 20000.0                     │
├───────────────

In [153]:
filtered_3_stl_camp_phone.loc[
    (filtered_3_stl_camp_phone['3m_limit_increase'] == 1) & 
    (filtered_3_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'] - filtered_3_stl_camp_phone['limit_previous_3m'] >= 500),
    ['CIF_ID', 'CUST_3M_TENURE', 'LIMIT_3M_TENURE_ADJUSTED', 'OPT_IN', 'PREFERREDPHONE']
].to_excel('19_Mar_2026_3mths_lim_inc.xlsx', index=False)


filtered_stl_camp_phone.loc[
    (filtered_stl_camp_phone['limit_increase'] == 1) &
    (filtered_stl_camp_phone['ADJUSTED_LIMIT'] - filtered_stl_camp_phone['REDIS_LIMIT'] >= 500),
    ['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE']
].to_excel('19_Mar_2026_1mths_lim_inc.xlsx', index=False)


filtered_3_stl_camp_phone.loc[
    (filtered_3_stl_camp_phone['3m_new_limits'] == 1),
    ['CIF_ID', 'CUST_3M_TENURE', 'LIMIT_3M_TENURE_ADJUSTED', 'OPT_IN', 'PREFERREDPHONE']
].to_excel('19_Mar_2026_3mths_lim_new.xlsx', index=False)


filtered_stl_camp_phone.loc[
    (filtered_stl_camp_phone['new_limits'] == 1),
    ['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE']
].to_excel('19_Mar_2026_1mths_lim_new.xlsx', index=False)

In [159]:
from tabulate import tabulate

# Filtered datasets
df_3m_lim_inc = filtered_3_stl_camp_phone.loc[
    (filtered_3_stl_camp_phone['3m_limit_increase'] == 1) & 
    (filtered_3_stl_camp_phone['LIMIT_3M_TENURE_ADJUSTED'] - filtered_3_stl_camp_phone['limit_previous_3m'] >= 500),
    ['CIF_ID', 'CUST_3M_TENURE', 'LIMIT_3M_TENURE_ADJUSTED', 'OPT_IN', 'PREFERREDPHONE']
]

df_1m_lim_inc = filtered_stl_camp_phone.loc[
    (filtered_stl_camp_phone['limit_increase'] == 1) &
    (filtered_stl_camp_phone['ADJUSTED_LIMIT'] - filtered_stl_camp_phone['REDIS_LIMIT'] >= 500),
    ['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE']
]

df_3m_lim_new = filtered_3_stl_camp_phone.loc[
    (filtered_3_stl_camp_phone['3m_new_limits'] == 1),
    ['CIF_ID', 'CUST_3M_TENURE', 'LIMIT_3M_TENURE_ADJUSTED', 'OPT_IN', 'PREFERREDPHONE']
]

df_1m_lim_new = filtered_stl_camp_phone.loc[
    (filtered_stl_camp_phone['new_limits'] == 1),
    ['CIF_ID', 'ADJUSTED_LIMIT', 'OPT_IN', 'PREFERREDPHONE']
]

# EDA summary table
eda_summary = pd.DataFrame({
    'Metric': [
        'Total Customers',
        #'Unique Phone Numbers',
        'Unique CIFs',
        'Opt-ins',
        'Sum of Limits (Millions)',
        'Avg Limit',
        'Median Limit',
        'Min Limit',
        'Max Limit'
    ],
    '3-Month Limit Increase': [
        len(df_3m_lim_inc),
        #df_3m_lim_inc['PREFERREDPHONE'].nunique(),
        df_3m_lim_inc['CIF_ID'].nunique(),
        df_3m_lim_inc['OPT_IN'].sum() if df_3m_lim_inc['OPT_IN'].dtype != 'object' else df_3m_lim_inc['OPT_IN'].value_counts().get('Y', 0),
        round(df_3m_lim_inc['LIMIT_3M_TENURE_ADJUSTED'].sum() / 1000000, 2),
        round(df_3m_lim_inc['LIMIT_3M_TENURE_ADJUSTED'].mean(), 2),
        round(df_3m_lim_inc['LIMIT_3M_TENURE_ADJUSTED'].median(), 2),
        round(df_3m_lim_inc['LIMIT_3M_TENURE_ADJUSTED'].min(), 2),
        round(df_3m_lim_inc['LIMIT_3M_TENURE_ADJUSTED'].max(), 2),
    ],
    '1-Month Limit Increase': [
        len(df_1m_lim_inc),
        #df_1m_lim_inc['PREFERREDPHONE'].nunique(),
        df_1m_lim_inc['CIF_ID'].nunique(),
        df_1m_lim_inc['OPT_IN'].sum() if df_1m_lim_inc['OPT_IN'].dtype != 'object' else df_1m_lim_inc['OPT_IN'].value_counts().get('Y', 0),
        round(df_1m_lim_inc['ADJUSTED_LIMIT'].sum() / 1000000, 2),
        round(df_1m_lim_inc['ADJUSTED_LIMIT'].mean(), 2),
        round(df_1m_lim_inc['ADJUSTED_LIMIT'].median(), 2),
        round(df_1m_lim_inc['ADJUSTED_LIMIT'].min(), 2),
        round(df_1m_lim_inc['ADJUSTED_LIMIT'].max(), 2),
    ],
    '3-Month New Limits': [
        len(df_3m_lim_new),
        #df_3m_lim_new['PREFERREDPHONE'].nunique(),
        df_3m_lim_new['CIF_ID'].nunique(),
        df_3m_lim_new['OPT_IN'].sum() if df_3m_lim_new['OPT_IN'].dtype != 'object' else df_3m_lim_new['OPT_IN'].value_counts().get('Y', 0),
        round(df_3m_lim_new['LIMIT_3M_TENURE_ADJUSTED'].sum() / 1000000, 2),
        round(df_3m_lim_new['LIMIT_3M_TENURE_ADJUSTED'].mean(), 2),
        round(df_3m_lim_new['LIMIT_3M_TENURE_ADJUSTED'].median(), 2),
        round(df_3m_lim_new['LIMIT_3M_TENURE_ADJUSTED'].min(), 2),
        round(df_3m_lim_new['LIMIT_3M_TENURE_ADJUSTED'].max(), 2),
    ],
    '1-Month New Limits': [
        len(df_1m_lim_new),
        #df_1m_lim_new['PREFERREDPHONE'].nunique(),
        df_1m_lim_new['CIF_ID'].nunique(),
        df_1m_lim_new['OPT_IN'].sum() if df_1m_lim_new['OPT_IN'].dtype != 'object' else df_1m_lim_new['OPT_IN'].value_counts().get('Y', 0),
        round(df_1m_lim_new['ADJUSTED_LIMIT'].sum() / 1000000, 2),
        round(df_1m_lim_new['ADJUSTED_LIMIT'].mean(), 2),
        round(df_1m_lim_new['ADJUSTED_LIMIT'].median(), 2),
        round(df_1m_lim_new['ADJUSTED_LIMIT'].min(), 2),
        round(df_1m_lim_new['ADJUSTED_LIMIT'].max(), 2),
    ]
})

# Convert to tabulate format with fancy_grid
print(tabulate(eda_summary, headers='keys', tablefmt='fancy_grid', showindex=False))

# Or if you want to save directly to a file:
eda_summary.to_csv('eda_summary.csv', index=False)
print("Data saved to 'eda_summary.csv'")

╒══════════════════════════╤══════════════════════════╤══════════════════════════╤══════════════════════╤══════════════════════╕
│ Metric                   │   3-Month Limit Increase │   1-Month Limit Increase │   3-Month New Limits │   1-Month New Limits │
╞══════════════════════════╪══════════════════════════╪══════════════════════════╪══════════════════════╪══════════════════════╡
│ Total Customers          │                        0 │                        0 │               238    │              6016    │
├──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────────┼──────────────────────┤
│ Unique CIFs              │                        0 │                        0 │               238    │              6016    │
├──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────────┼──────────────────────┤
│ Opt-ins                  │                        0 │                        0 │               

In [214]:
filtered_3_stl_camp_phone.head()

,CIF_ID,CUST_AGE,AGE_WITH_BANK,PREDICTIONS,RISK_GRADE,CREDIT_SCORE,PROXY_INCOME,PROXY_INCOME_NEW,PROXY_INCOME_MSE,NUMBER_OF_20K_IN_4MONTHS,COMPUTED_LIMIT,LOYALTY_BAND,NEXT_LOYALTY_BAND,HYBRID_RISK_GRADE,HYBRID_SCORE,SEGMENT,PROD_TENURE_UNSECURED_LONG_TERM,PROD_TENURE_GLOBAL_LIMIT_MARGIN_DSR,PROD_TENURE_UNSECURED_STL_DSR_MARGIN,PROD_TENURE_OD_PRODUCT,PROD_TENURE_BNPL_SHORT_TERM,PROD_TENURE_BNPL_LONG_TERM,D_I_RATIO_UNSECURED_LONG_TERM,D_I_RATIO_GLOBAL_LIMIT_MARGIN_DSR,D_I_RATIO_UNSECURED_STL_DSR_MARGIN,D_I_RATIO_OD_PRODUCT,D_I_RATIO_BNPL_SHORT_TERM,D_I_RATIO_BNPL_LONG_TERM,MIN_LIMIT_UNSECURED_LONG_TERM,MIN_LIMIT_GLOBAL_LIMIT_MARGIN_DSR,MIN_LIMIT_UNSECURED_STL_DSR_MARGIN,MIN_LIMIT_OD_PRODUCT,MIN_LIMIT_BNPL_SHORT_TERM,MIN_LIMIT_BNPL_LONG_TERM,MAX_LIMIT_UNSECURED_LONG_TERM,MAX_LIMIT_GLOBAL_LIMIT_MARGIN_DSR,MAX_LIMIT_UNSECURED_STL_DSR_MARGIN,MAX_LIMIT_OD_PRODUCT,MAX_LIMIT_BNPL_SHORT_TERM,MAX_LIMIT_BNPL_LONG_TERM,BNPL_INTEREST_RATE,UPL_INTEREST_RATE,CUSTOMER_TYPE,PRODUCT_INTERACTION,CUSTOMER_RELATIONSHIP,CREDIT_HISTORY,MAX_DPD,MAX_DPD_CRB,SEGMENTATION_CLASS,CRB_BORROWERS,UPL_CONSERVATIVE_TENOR,UPL_INTERMEDIATE_TENOR,CREDIT_MARGIN,UPL_CONSERVATIVE_DSR,UPL_CONSERVATIVE_LIMIT,UPL_INTERMEDIATE_DSR,UPL_INTERMEDIATE_LIMIT,UPL_CONSERVATIVE_INTEREST_RATE,UPL_CONSERVATIVE_TENOR_MARGIN,UPL_INTERMEDIATE_TENOR_MARGIN,ADJUSTED_LIMIT,INTEREST_FEE,FACILITY_FEE,CRB_NON_MOBILE_LOAN_HISTORY,INTERNAL_LOAN_HISTORY,FIRST_ACCT_OPN_DATE,LAST_CLOSED_STL_OPN_DATE,MONTHS_ACTIVE_STL,MONTHS_STL_L6M,OTGREGISTERED,LAST_LOAN_OPN_DATE,LAST_LOAN_CLS_DATE,LAST_LOAN_DPD,MAX_STL_DPD,UPL_LIMIT,TAKEN_UPL,MAX_DPD_ALL_LOANS,CLOSED_STL_OPENED>3M,LAST_OPN_LOAN<3M,LAST_CLS_LOAN<3M,PERFORMING_INTERNALLY,AGE_WITH_BANK_MONTHS,NTB,ARREARS0DAYS,ARREARS30DAYS,ARREARS60DAYS,ARREARS90DAYS,ARREARSGT90DAYS,NONPERFORMING,STL_ACTIVE_LOAN,TAKEN_STL_LOAN,STL_HISTORY,LATEST_UTILIZED_LIMIT,ID,OPT_IN,OPT_IN_DATE,REDIS_LIMIT,BLACKLISTED,CREDIT_LIFE_INSURANCE,EXCISE_DUTY,PHCF,REFRESH_DATE,NTB_LIMIT,NTB_ADJUSTED_LIMIT,CUST_3M_TENURE,LIMIT_3M_TENURE,LIMIT_3M_TENURE_ADJUSTED,BNPL_MONTHLY_LIMIT_REDIS,BNPL_TOTAL_LIMIT_REDIS,UPL_MONTHLY_STP_LIMIT,UPL_TOTAL_STP_LIMIT,UPL_MONTHLY_NON_STP_LIMIT,UPL_TOTAL_NON_STP_LIMIT,BNPL_MONTHLY_LIMIT,BNPL_TOTAL_LIMIT,OTG_RET_LOGIN,MSE_NON_SOLOBIZ,STL_PRICE_CAMPAIGN,BATCH_NO,new_limits,limit_increase,limit_cancelled,limit_decrease,bnpl_new_limits,bnpl_limit_increase,bnpl_limit_cancelled,bnpl_limit_decrease,solo_flag,previous_3m,limit_previous_3m,3m_new_limits,3m_limit_increase,3m_limit_cancelled,3m_limit_decrease,PREFERREDPHONE,SEGMENTCODE,info
1399,0622840,33,1,1,HIGH,371.5,17606315.25,17606315.25,4650929.25,4,40000.0,Silver,Gold,MEDIUM,589.0,Premium Credit,48,48,1,1,6,12,0.3,0.35,0.35,0.35,0.35,0.21,50000,50000,1000,5000,5000,20000,1500000,1500000,100000,100000,100000,250000,0.02,NaN,Premium,Medium,High,Strong,0,0,3100,Strong,12,24,4.0,0.15,500000,0.25,1500000,0.198,0.5,0.75,40000.0,0.115,0.115,YES,YES,2025-01-24,2025-12-23,8.0,3.0,Y,2026-02-08,NaN,0.0,36.0,0.0,NaN,0,N,Y,N,Y,12,N,48,0,0,0,0,0,YES,YES,YES,40000.0,133104.0,1,2025-01-25 10:00:00,40000.0,N,0.0006,0.2,0.0025,2026-02-13,6000,0.0,1.0,52013.20088877261150628,150000.0,0.0,0,2640947.28750000009313226,500000.0,4401578.8125,1500000.0,0.0,0,Y,N,0,202601,0,0,0,0,0,0,0,0,1,0.0,0.0,1,0,0,0,254790023526.0,3100,both
1447,0422759,25,2,1,HIGH,532.0,90050.0,90050.0,63475.0,4,27015.0,Platinum,Platinum,LOW,743.0,Standard Credit,24,24,1,1,6,6,0.25,0.3,0.3,0.3,0.3,0.18,50000,50000,1000,5000,5000,20000,500000,500000,60000,60000,60000,200000,0.02,NaN,Non_Premium,Medium,Medium,Strong,0,0,3100,Strong,9,18,4.0,0.15,200000,0.2,1000000,0.193,0.0,0.5,23447.0,0.075,0.075,YES,YES,2023-09-14,2025-09-12,18.0,1.0,Y,2026-02-09,NaN,0.0,31.0,0.0,NaN,0,Y,Y,N,Y,29,N,99,0,0,0,0,0,NO,YES,YES,29449.0,47986.0,1,2024-04-08 15:20:52,23447.0,N,0.0006,0.2,0.0025,2026-02-13,6000,0.0,1.0,13505.0542334335113992,38947.0,0.0,0,13507.5,112930.0,18010.0,282263.0,0.0,0,Y,N,0,202601,0,0,0,0,0,0,0,0,0,1.0,38947.0,0,0,0,0,254742449805.0,3100,both
1470,0379188,

In [216]:
filtered_3_stl_camp_phone[['CIF_ID', 'ADJUSTED_LIMIT','OPT_IN','PREFERREDPHONE']].head()

,CIF_ID,ADJUSTED_LIMIT,OPT_IN,PREFERREDPHONE
1399,0622840,40000.0,1,254790023526.0
1447,0422759,23447.0,1,254742449805.0
1470,0379188,100000.0,1,254112322217.0
1493,0319292,100000.0,1,+254727089166
1554,0514330,38657.0,1,+254707170905


##### STL Discount Campaign 

In [218]:
query_camp = """
select * from analytics.stl_1m_pricing_discount_target
"""

camp_1month_df = pd.read_sql(query_camp, con=connection_prod, dtype={'CIF_ID': str})

camp_1month_df.to_csv('camp_1month_df.csv', index = False)


/var/folders/_m/ry3gsj516338hx1t402h2qd1_l3rdz/T/ipykernel_20309/1639186518.py:5: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [220]:
stl_discount_df = pd.read_csv('camp_1month_df.csv', dtype={'CIF_ID': str})

merged_df['DISCOUNT_FLAG'] = merged_df['CIF_ID'].isin(stl_discount_df['CIF_ID']).map({True: 'Y', False: 'N'})


In [222]:
filtered_stl_discount = merged_df[
    (merged_df['ADJUSTED_LIMIT'] > 0) #& (merged_df['MSE_NON_SOLOBIZ'] == 'N') & (merged_df['NTB'] == 'N') 
& (merged_df['DISCOUNT_FLAG'] == 'Y')]

filtered_stl_discount_phone = filtered_stl_discount.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 'SEGMENTCODE']], on='CIF_ID',how='inner',indicator='info')

filtered_stl_discount_phone[['CIF_ID', 'ADJUSTED_LIMIT','OPT_IN','PREFERREDPHONE', 'FACILITY_FEE', 'LOYALTY_BAND']] \
    .to_excel('1_stl_sep_02_camp_Sep_16_v3.xlsx', index=False)

filtered_3_stl_camp_phone = filtered_stl_discount_phone[(filtered_stl_discount_phone['ADJUSTED_LIMIT'] > 0) & (filtered_stl_discount_phone['CUST_3M_TENURE'] == 1 )&
                                                                            (filtered_stl_discount_phone['LIMIT_3M_TENURE_ADJUSTED'] > 0) 
]

filtered_3_stl_camp_phone[['CIF_ID', 'CUST_3M_TENURE','LIMIT_3M_TENURE_ADJUSTED','OPT_IN','PREFERREDPHONE', 'FACILITY_FEE','LOYALTY_BAND']] \
    .to_excel('3_stl_sep_16_camp_Sep_16_v3.xlsx', index=False)


###### BNPL staff

In [224]:
filtered_bnpl = merged_df[(merged_df['BNPL_TOTAL_LIMIT'] > 0)]

In [226]:
filtered_bnpl = merged_df[(merged_df['BNPL_TOTAL_LIMIT'] > 0)]
filtered_bnpl_phone = filtered_bnpl.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 'SEGMENTCODE']], on='CIF_ID',how='inner',indicator='info')

filtered_bnpl_phone[['CIF_ID', 'BNPL_TOTAL_LIMIT','OPT_IN','PREFERREDPHONE']] \
    .to_excel('2_stl_sep_16_bnpl_25_v2.xlsx', index=False)



##### CRB and Income Extract

In [228]:
refresh_df[['CIF_ID', 'CUST_AGE','AGE_WITH_BANK','MAX_DPD_CRB','VALUE_CRB_LOAN_TAKEN',
           'VOLUME_CRB_LOAN_TAKEN', 'RISK_GRADE','CREDIT_SCORE','PROXY_INCOME','IM_LOAN_VALUE',
 'IM_BASKET_SHARE']].to_excel('cust_crb_score_risk_sep_25_V2.xlsx', index=False)

In [ ]:
#stl_cust_limsep26 = pd.read_excel('STL CIFS.xlsx', dtype={'CIF_ID': str})

query_open = """
SELECT C.CIF_ID, G.ACID, G.FORACID, 
    --FLOOR((SYSDATE -C.DATE_OF_BIRTH)/365) CUST_AGE, FLOOR((SYSDATE - C.CUST_OPN_DATE)/365) AGE_WITH_BANK, C.SEGMENTATION_CLASS, M.CUSTOMERINCOME, G.ACCT_CLS_FLG,staff_flag,
---M.SECTOR, M.SUBSECTOR, S.SECURED_FLG,M.PHYSICALADDRESS, 
---M.MARITALSTATUS, 
--M.OTG_TRANSACTING, 
    G.SCHM_CODE,
---S.SCHM_DESC, 
L.DIS_AMT, (G.CLR_BAL_AMT*-1) OUTSTANDING_AMOUNT,
LHT.APPLICABLE_DATE DISB_DATE
---CASE WHEN L.REP_PERD_MTHS = 0 THEN L.REP_PERD_DAYS/30 ELSE REP_PERD_MTHS END REP_PERD_MTHS, LRS.NUM_OF_FLOWS, GAC.DPD_CNTR,NVL(SUM(LRS.FLOW_AMT), 0)FLOW_AMT,  
--CASE WHEN (CASE WHEN L.REP_PERD_MTHS = 0 THEN L.REP_PERD_DAYS/30 ELSE REP_PERD_MTHS END - LRS.NUM_OF_FLOWS) <= 0 THEN LRS.NUM_OF_FLOWS
     --ELSE CASE WHEN L.REP_PERD_MTHS = 0 THEN L.REP_PERD_DAYS/30 ELSE REP_PERD_MTHS END - LRS.NUM_OF_FLOWS END NO_OF_PAID_INSTALLMENTS, G.ACCT_CRNCY_CODE, G.ACCT_CLS_DATE --, CBCD.CIPSCORE
     --,e.NRML_INTEREST_AMOUNT_DR 
FROM TBAADM.GAM@FINACLE G
LEFT JOIN TBAADM.CMG@FINACLE C on G.CIF_ID = C.CIF_ID
--LEFT JOIN EDW.MDM_IMKE M on G.CIF_ID = M.CUSTOMERID
--LEFT JOIN TBAADM.GSP@FINACLE S on G.SCHM_CODE = S.SCHM_CODE
LEFT JOIN TBAADM.LAM@FINACLE L on G.ACID = L.ACID
--LEFT JOIN TBAADM.GAC@FINACLE GAC ON G.ACID = GAC.ACID
--LEFT JOIN TBAADM.LRS@FINACLE LRS ON G.ACID = LRS.ACID
LEFT JOIN TBAADM.LHT@FINACLE LHT ON G.ACID = LHT.ACID
--LEFT JOIN tbaadm.eit@finacle e ON G.ACID = e.ENTITY_ID
--INMPROD.CREDIT_BUREAU_CIP_DATA@DGLENDING CBCD
WHERE G.SCHM_CODE IN ('LAL34','LAL47') 
AND L.DIS_AMT > 1000 
---AND LHT.APPLICABLE_DATE >= '21-OCT-2025'
---AND L.DIS_AMT >= 20000 
--AND L.REP_PERD_MTHS = 3
--group by C.CIF_ID, G.ACID,  G.FORACID,C.DATE_OF_BIRTH, C.CUST_OPN_DATE, C.SEGMENTATION_CLASS, M.CUSTOMERINCOME,G.ACCT_CLS_FLG, M.SECTOR, 
--M.SUBSECTOR, S.SECURED_FLG,M.PHYSICALADDRESS, M.MARITALSTATUS, 
--M.OTG_TRANSACTING, G.SCHM_CODE, 
--S.SCHM_DESC, 
--L.DIS_AMT, G.CLR_BAL_AMT, LHT.APPLICABLE_DATE, L.REP_PERD_MTHS, GAC.DPD_CNTR, LRS.NUM_OF_FLOWS, L.REP_PERD_DAYS, G.ACCT_CRNCY_CODE,staff_flag, G.ACCT_CLS_DATE

"""

#### Execute query and load into DataFrame ###
stl_cust_credits = pd.read_sql(query_open, con=connection, dtype={'CIF_ID': str})

stl_cust_credits.to_csv('stl_cust_credits.csv', index = False)

stl_cust_credits = pd.read_csv('stl_cust_credits.csv', dtype={'CIF_ID': str})



In [ ]:
cust_info_df.rename(columns={'CUSTOMERID': 'CIF_ID'}, inplace = True)

In [ ]:
#stl_cust_limsep26 = pd.read_excel('STL CIFS.xlsx', dtype={'CIF_ID': str})

query_open = """
SELECT C.CIF_ID, G.ACID, G.FORACID, 
    --FLOOR((SYSDATE -C.DATE_OF_BIRTH)/365) CUST_AGE, FLOOR((SYSDATE - C.CUST_OPN_DATE)/365) AGE_WITH_BANK, C.SEGMENTATION_CLASS, M.CUSTOMERINCOME, G.ACCT_CLS_FLG,staff_flag,
---M.SECTOR, M.SUBSECTOR, S.SECURED_FLG,M.PHYSICALADDRESS, 
---M.MARITALSTATUS, 
--M.OTG_TRANSACTING, 
    G.SCHM_CODE,
---S.SCHM_DESC, 
L.DIS_AMT, (G.CLR_BAL_AMT*-1) OUTSTANDING_AMOUNT,
LHT.APPLICABLE_DATE DISB_DATE
---, CASE WHEN L.REP_PERD_MTHS = 0 THEN L.REP_PERD_DAYS/30 ELSE REP_PERD_MTHS END REP_PERD_MTHS, LRS.NUM_OF_FLOWS, GAC.DPD_CNTR,NVL(SUM(LRS.FLOW_AMT), 0)FLOW_AMT,  
--CASE WHEN (CASE WHEN L.REP_PERD_MTHS = 0 THEN L.REP_PERD_DAYS/30 ELSE REP_PERD_MTHS END - LRS.NUM_OF_FLOWS) <= 0 THEN LRS.NUM_OF_FLOWS
     --ELSE CASE WHEN L.REP_PERD_MTHS = 0 THEN L.REP_PERD_DAYS/30 ELSE REP_PERD_MTHS END - LRS.NUM_OF_FLOWS END NO_OF_PAID_INSTALLMENTS, G.ACCT_CRNCY_CODE, G.ACCT_CLS_DATE --, CBCD.CIPSCORE
     --,e.NRML_INTEREST_AMOUNT_DR 
FROM TBAADM.GAM@FINACLE G
LEFT JOIN TBAADM.CMG@FINACLE C on G.CIF_ID = C.CIF_ID
--LEFT JOIN EDW.MDM_IMKE M on G.CIF_ID = M.CUSTOMERID
--LEFT JOIN TBAADM.GSP@FINACLE S on G.SCHM_CODE = S.SCHM_CODE
LEFT JOIN TBAADM.LAM@FINACLE L on G.ACID = L.ACID
--LEFT JOIN TBAADM.GAC@FINACLE GAC ON G.ACID = GAC.ACID
--LEFT JOIN TBAADM.LRS@FINACLE LRS ON G.ACID = LRS.ACID
LEFT JOIN TBAADM.LHT@FINACLE LHT ON G.ACID = LHT.ACID
--LEFT JOIN tbaadm.eit@finacle e ON G.ACID = e.ENTITY_ID
--INMPROD.CREDIT_BUREAU_CIP_DATA@DGLENDING CBCD
WHERE G.SCHM_CODE IN ('LAL34','LAL47') 
AND L.DIS_AMT > 1000 
AND LHT.APPLICABLE_DATE >= '30-NOV-2025' AND LHT.APPLICABLE_DATE < '01-JAN-2026'
---AND L.DIS_AMT >= 20000 
--AND L.REP_PERD_MTHS = 3
--group by C.CIF_ID, G.ACID,  G.FORACID,C.DATE_OF_BIRTH, C.CUST_OPN_DATE, C.SEGMENTATION_CLASS, M.CUSTOMERINCOME,G.ACCT_CLS_FLG, M.SECTOR, 
--M.SUBSECTOR, S.SECURED_FLG,M.PHYSICALADDRESS, M.MARITALSTATUS, 
--M.OTG_TRANSACTING, G.SCHM_CODE, 
--S.SCHM_DESC, 
--L.DIS_AMT, G.CLR_BAL_AMT, LHT.APPLICABLE_DATE, L.REP_PERD_MTHS, GAC.DPD_CNTR, LRS.NUM_OF_FLOWS, L.REP_PERD_DAYS, G.ACCT_CRNCY_CODE,staff_flag, G.ACCT_CLS_DATE

"""

#### Execute query and load into DataFrame ###
stl_cust_credits_dec = pd.read_sql(query_open, con=connection, dtype={'CIF_ID': str})

stl_cust_credits_dec.to_csv('stl_cust_credits_dec.csv', index = False)

stl_cust_credits_dec = pd.read_csv('stl_cust_credits_dec.csv', dtype={'CIF_ID': str})



In [ ]:
stl_cust_credits = pd.read_csv('stl_cust_credits.csv', dtype={'CIF_ID': str})

stl_cust_credits = stl_cust_credits.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 'SEGMENTCODE', 'ID_BRANCH', 'BRANCH_NAME', 'SECTOR', 'SUBSECTOR' ]], on='CIF_ID'
    ,how='inner',indicator='info')

In [ ]:
stl_cust_credits_dec = pd.read_csv('stl_cust_credits_dec.csv', dtype={'CIF_ID': str})

stl_cust_credits_dec = stl_cust_credits_dec.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 'SEGMENTCODE', 'ID_BRANCH', 'BRANCH_NAME', 'SECTOR', 'SUBSECTOR', 'OCCUPATION' ]], on='CIF_ID'
    ,how='inner',indicator='info')

In [ ]:
stl_cust_credits[['CIF_ID', 'ACID', 'FORACID','DISB_DATE', 'DIS_AMT' , 'OUTSTANDING_AMOUNT', 'ID_BRANCH', 'BRANCH_NAME']] \
    .to_excel('stl_cust_credits_branch_20Jan26.xlsx', index=False)

In [ ]:
stl_cust_credits_dec[['CIF_ID', 'ACID', 'FORACID', 'DISB_DATE', 'DIS_AMT' , 'OUTSTANDING_AMOUNT', 'ID_BRANCH', 'BRANCH_NAME', 'SECTOR', 'SUBSECTOR', 'OCCUPATION']] \
    .to_excel('stl_cust_credits_dec_occupation.xlsx', index=False)

In [ ]:
stl_cust_limits = merged_df.merge(
    cust_info_df[['CIF_ID', 'PREFERREDPHONE', 'SEGMENTCODE', 'ID_BRANCH', 'BRANCH_NAME', 'SECTOR', 'SUBSECTOR' ]], on='CIF_ID'
    ,how='inner',indicator='info')

In [ ]:
stl_cust_limits[['CIF_ID', 'ADJUSTED_LIMIT', 'BATCH_NO', 'ID_BRANCH', 'BRANCH_NAME', 'SECTOR', 'SUBSECTOR',
                 'PROXY_INCOME','PROXY_INCOME_NEW', 'PROXY_INCOME_MSE']] \
    .to_excel('stl_cust_limits_branch_proxy_2Feb26.xlsx', index=False)

In [ ]:
stl_cust_limits.columns.to_list()

In [ ]:
cust_info_df.columns.to_list()